Validate Datas Structure S Files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install duckdb

import os
import duckdb
import pandas as pd

BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"

H0_BASELINE_YEAR = 1992  # pick an H0 year you know exists
S2_START_YEAR, S2_END_YEAR = 1976, 2023

# Adjust if your columns differ
EXPECTED_COLS = ["exporter", "importer", "commoditycode", "value_exporter", "value_importer"]

def describe_parquet(con, path):
    # schema-only; does not scan the full file
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()

def quick_sample(con, path, n=5):
    return con.execute(f"SELECT * FROM read_parquet('{path}') LIMIT {n}").df()

con = duckdb.connect()

# -------------------------
# 1) Get H0 baseline schema
# -------------------------
h0_path = os.path.join(BASE, f"H0_{H0_BASELINE_YEAR}.parquet")
if not os.path.exists(h0_path):
    raise FileNotFoundError(f"Baseline H0 file not found: {h0_path}")

h0_sch = describe_parquet(con, h0_path)
h0_types = h0_sch.set_index("column_name")["column_type"].to_dict()
h0_cols = h0_sch["column_name"].tolist()

print("H0 baseline file:", h0_path)
print("H0 baseline #cols:", len(h0_cols))
print("H0 baseline expected-cols present?:", all(c in set(h0_cols) for c in EXPECTED_COLS))

# -------------------------
# 2) Check S2 files vs H0
# -------------------------
rows = []
missing_files = []
missing_cols = []
type_mismatches = []
extra_cols = []
missing_relative_to_h0 = []

for y in range(S2_START_YEAR, S2_END_YEAR + 1):
    s2_path = os.path.join(BASE, f"S2_{y}.parquet")
    if not os.path.exists(s2_path):
        missing_files.append(y)
        rows.append({"year": y, "status": "missing_file"})
        continue

    try:
        s2_sch = describe_parquet(con, s2_path)
        s2_cols = s2_sch["column_name"].tolist()
        s2_types = s2_sch.set_index("column_name")["column_type"].to_dict()

        # required columns check
        miss_req = [c for c in EXPECTED_COLS if c not in set(s2_cols)]
        if miss_req:
            missing_cols.append((y, miss_req))

        # compare to H0 baseline: missing/extra columns (strict structure check)
        miss_vs_h0 = [c for c in h0_cols if c not in set(s2_cols)]
        extra_vs_h0 = [c for c in s2_cols if c not in set(h0_cols)]
        if miss_vs_h0:
            missing_relative_to_h0.append((y, miss_vs_h0))
        if extra_vs_h0:
            extra_cols.append((y, extra_vs_h0))

        # type mismatches for shared columns
        mism = []
        for c in set(h0_cols).intersection(set(s2_cols)):
            if h0_types.get(c) != s2_types.get(c):
                mism.append((c, h0_types.get(c), s2_types.get(c)))
        if mism:
            type_mismatches.append((y, mism))

        # optional: quick sanity on a small sample (won’t fail the run)
        # comment out if you want schema-only
        smp = quick_sample(con, s2_path, n=5)

        rows.append({
            "year": y,
            "status": "ok",
            "n_cols": len(s2_cols),
            "has_expected_cols": (len(miss_req) == 0),
            "missing_vs_h0": len(miss_vs_h0),
            "extra_vs_h0": len(extra_vs_h0),
            "type_mismatch_count": len(mism),
        })

    except Exception as e:
        rows.append({"year": y, "status": "error", "error": str(e)})

con.close()

report = pd.DataFrame(rows).sort_values("year")

print("\nMissing S2 files:", missing_files[:30], "..." if len(missing_files) > 30 else "")
print("\nYears missing REQUIRED cols (first 10):")
print(missing_cols[:10], "..." if len(missing_cols) > 10 else "")
print("\nYears with TYPE mismatches vs H0 (first 5):")
print(type_mismatches[:5], "..." if len(type_mismatches) > 5 else "")
print("\nYears with columns missing relative to H0 baseline (first 5):")
print(missing_relative_to_h0[:5], "..." if len(missing_relative_to_h0) > 5 else "")
print("\nYears with extra columns relative to H0 baseline (first 5):")
print(extra_cols[:5], "..." if len(extra_cols) > 5 else "")

display(report)


Mounted at /content/drive
H0 baseline file: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/H0_1992.parquet
H0 baseline #cols: 7
H0 baseline expected-cols present?: True

Missing S2 files: [] 

Years missing REQUIRED cols (first 10):
[] 

Years with TYPE mismatches vs H0 (first 5):
[] 

Years with columns missing relative to H0 baseline (first 5):
[] 

Years with extra columns relative to H0 baseline (first 5):
[] 


,year,status,n_cols,has_expected_cols,missing_vs_h0,extra_vs_h0,type_mismatch_count
0,1976,ok,7,True,0,0,0
1,1977,ok,7,True,0,0,0
2,1978,ok,7,True,0,0,0
3,1979,ok,7,True,0,0,0
4,1980,ok,7,True,0,0,0
5,1981,ok,7,True,0,0,0
6,1982,ok,7,True,0,0,0
7,1983,ok,7,True,0,0,0
8,1984,ok,7,True,0,0,0
9,1985,ok,7,True,0,0,0


Validate H and S files compatibility

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install duckdb

import os
import duckdb
import pandas as pd

BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"

# Pick representative years (1 early, 1 mid, 1 late) for each family
H0_YEARS = [1992, 2000, 2023]
S2_YEARS = [1976, 2000, 2023]

# Columns we care about for the Moran pipeline
KEY_COLS = ["exporter", "importer", "commoditycode", "value_exporter", "value_importer"]

def describe(con, path):
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()

def sample(con, path, n=5):
    return con.execute(f"SELECT * FROM read_parquet('{path}') LIMIT {n}").df()

def summarize_schema(df_desc):
    # df_desc columns: column_name, column_type, null, key, default, extra
    out = df_desc[["column_name", "column_type"]].copy()
    out["column_name"] = out["column_name"].astype(str)
    out["column_type"] = out["column_type"].astype(str)
    return out

def keycol_report(df_desc, label):
    types = df_desc.set_index("column_name")["column_type"].to_dict()
    rows = []
    for c in KEY_COLS:
        rows.append({
            "dataset": label,
            "column": c,
            "present": c in types,
            "type": types.get(c, None),
        })
    return pd.DataFrame(rows)

con = duckdb.connect()

# -----------------------
# 1) Schema for selected years
# -----------------------
schema_rows = []
key_reports = []

for y in H0_YEARS:
    fp = os.path.join(BASE, f"H0_{y}.parquet")
    d = describe(con, fp)
    s = summarize_schema(d)
    s["dataset"] = "H0"
    s["year"] = y
    schema_rows.append(s)

    key_reports.append(keycol_report(d, f"H0_{y}"))

for y in S2_YEARS:
    fp = os.path.join(BASE, f"S2_{y}.parquet")
    d = describe(con, fp)
    s = summarize_schema(d)
    s["dataset"] = "S2"
    s["year"] = y
    schema_rows.append(s)

    key_reports.append(keycol_report(d, f"S2_{y}"))

schema_df = pd.concat(schema_rows, ignore_index=True)
key_df = pd.concat(key_reports, ignore_index=True)

print("=== Key columns presence/types (this is what we need to adapt the script) ===")
display(key_df)

# -----------------------
# 2) Compare FULL column sets (optional but useful)
# -----------------------
def colset_for(con, path):
    d = describe(con, path)
    return d["column_name"].tolist()

h0_cols = colset_for(con, os.path.join(BASE, f"H0_{H0_YEARS[0]}.parquet"))
s2_cols = colset_for(con, os.path.join(BASE, f"S2_{S2_YEARS[0]}.parquet"))

print("H0 baseline #cols:", len(h0_cols))
print("S2 baseline #cols:", len(s2_cols))
print("Columns in H0 not in S2 (first 30):", [c for c in h0_cols if c not in s2_cols][:30])
print("Columns in S2 not in H0 (first 30):", [c for c in s2_cols if c not in h0_cols][:30])

# -----------------------
# 3) Tiny sample to confirm names & numeric behavior
# -----------------------
def quick_value_checks(con, path, label):
    # pull only key columns if present
    d = describe(con, path)
    cols = set(d["column_name"].tolist())
    keep = [c for c in KEY_COLS if c in cols]
    if not keep:
        return pd.DataFrame({"dataset":[label], "note":["No key columns found"]})

    q = "SELECT " + ", ".join(keep) + f" FROM read_parquet('{path}') LIMIT 10"
    smp = con.execute(q).df()

    out = {"dataset": label}
    for c in keep:
        out[f"{c}_dtype"] = str(smp[c].dtype)
        if c.startswith("value_"):
            out[f"{c}_min"] = float(pd.to_numeric(smp[c], errors="coerce").min())
            out[f"{c}_max"] = float(pd.to_numeric(smp[c], errors="coerce").max())
    return pd.DataFrame([out])

checks = []
checks.append(quick_value_checks(con, os.path.join(BASE, f"H0_{H0_YEARS[0]}.parquet"), f"H0_{H0_YEARS[0]}"))
checks.append(quick_value_checks(con, os.path.join(BASE, f"S2_{S2_YEARS[0]}.parquet"), f"S2_{S2_YEARS[0]}"))

print("\n=== Quick sample dtype/range checks (key columns only) ===")
display(pd.concat(checks, ignore_index=True))

con.close()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Key columns presence/types (this is what we need to adapt the script) ===


,dataset,column,present,type
0,H0_1992,exporter,True,VARCHAR
1,H0_1992,importer,True,VARCHAR
2,H0_1992,commoditycode,True,VARCHAR
3,H0_1992,value_exporter,True,DOUBLE
4,H0_1992,value_importer,True,DOUBLE
5,H0_2000,exporter,True,VARCHAR
6,H0_2000,importer,True,VARCHAR
7,H0_2000,commoditycode,True,VARCHAR
8,H0_2000,value_exporter,True,DOUBLE
9,H0_2000,value_importer,True,DOUBLE


H0 baseline #cols: 7
S2 baseline #cols: 7
Columns in H0 not in S2 (first 30): []
Columns in S2 not in H0 (first 30): []

=== Quick sample dtype/range checks (key columns only) ===


,dataset,exporter_dtype,importer_dtype,commoditycode_dtype,value_exporter_dtype,value_exporter_min,value_exporter_max,value_importer_dtype,value_importer_min,value_importer_max
0,H0_1992,object,object,object,float64,0.0,0.0,float64,2594.395356,36480.170943
1,S2_1976,object,object,object,float64,0.0,204533.0,float64,0.000000,179531.823173



Moran I and CI Estimation. Normality assumption


In [ ]:
# ============================================================
# Moran's I by year with NORMAL-ASSUMPTION inference only
# (no permutation "CI"; permutation p-value optional but OFF by default)
#
# Outputs for exports and imports, per year:
#   n, I, EI (expected I under null), VI_norm, SE_norm,
#   z_norm, p_norm (two-sided),
#   CI95_norm_low/high
#
# Works for either dataset family:
#   PREFIX="H0" with YEARS 1992–2023
#   PREFIX="S2" with YEARS 1976–2023
#
# Stand-alone: reads OD_Matrix.csv + {PREFIX}_YYYY.parquet
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install duckdb libpysal esda scipy

import os
import numpy as np
import pandas as pd
import duckdb

from esda.moran import Moran
from libpysal.weights import W as psW
from scipy.stats import norm

# ----------------------------
# Config (EDIT THESE)
# ----------------------------
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
OD_PATH = os.path.join(BASE, "OD_Matrix.csv")

PREFIX = "S2"          # "H0" or "S2"
START_YEAR = 1976      # H0: 1992 ; S2: 1976
END_YEAR   = 2023

ALPHA = 0.05           # 95% CI
PERMUTATIONS = 0       # set to 999 if you ALSO want p_sim/z_sim saved (optional)

EXPORTER_COL = "exporter"
IMPORTER_COL = "importer"
VALUE_EXPORT_COL = "value_exporter"
VALUE_IMPORT_COL = "value_importer"

OUT_CSV = os.path.join(BASE, f"moran_{PREFIX}_{START_YEAR}_{END_YEAR}_normal_inference.csv")

# ----------------------------
# Load OD once -> W_full once (inverse-distance, row-standardized)
# ----------------------------
od_long = pd.read_csv(OD_PATH)
od_long["origin"] = od_long["origin"].astype(str).str.upper()
od_long["destination"] = od_long["destination"].astype(str).str.upper()

D_full = od_long.pivot(index="origin", columns="destination", values="distance_km")
all_codes = sorted(set(D_full.index) | set(D_full.columns))
D_full = D_full.reindex(index=all_codes, columns=all_codes)

np.fill_diagonal(D_full.values, 0.0)

D = D_full.to_numpy(dtype=float)
np.fill_diagonal(D, np.nan)
W_full = 1.0 / D
W_full = W_full / np.nansum(W_full, axis=1, keepdims=True)

code_to_ix = {c: i for i, c in enumerate(all_codes)}

# ----------------------------
# Helpers
# ----------------------------
def log10_1p(x):
    return np.log10(np.asarray(x, dtype=float) + 1.0)

def subset_weights_dense(codes):
    idx = np.array([code_to_ix[c] for c in codes], dtype=int)
    W = W_full[np.ix_(idx, idx)].copy()
    np.fill_diagonal(W, 0.0)

    rs = W.sum(axis=1, keepdims=True)
    rs[rs == 0] = np.nan
    W = W / rs

    neighbors, weights = {}, {}
    for i, c in enumerate(codes):
        js = np.where(np.isfinite(W[i]) & (W[i] > 0))[0]
        neighbors[c] = [codes[j] for j in js]
        weights[c] = [float(W[i, j]) for j in js]

    w = psW(neighbors, weights)
    w.transform = "R"
    return w

def agg_year_trade(parquet_path: str):
    con = duckdb.connect()
    exports = con.execute(f"""
        SELECT UPPER({EXPORTER_COL}) AS ISO_A3, SUM({VALUE_EXPORT_COL}) AS exports
        FROM read_parquet('{parquet_path}')
        GROUP BY 1
    """).df()
    imports = con.execute(f"""
        SELECT UPPER({IMPORTER_COL}) AS ISO_A3, SUM({VALUE_IMPORT_COL}) AS imports
        FROM read_parquet('{parquet_path}')
        GROUP BY 1
    """).df()
    con.close()
    return exports, imports

def moran_normal_stats(codes, values, alpha=0.05, permutations=0):
    """
    Returns normal-assumption inference stats for Moran's I.
    If permutations>0, also returns p_sim and z_sim for reference.
    """
    values = np.asarray(values, dtype=float)
    codes = [str(c).upper() for c in codes]

    # drop non-finite
    mask = np.isfinite(values)
    codes = [c for c, ok in zip(codes, mask) if ok]
    values = values[mask]

    # keep only OD universe
    kept_codes, kept_vals = [], []
    for c, v in zip(codes, values):
        if c in code_to_ix:
            kept_codes.append(c)
            kept_vals.append(v)
    codes = kept_codes
    values = np.asarray(kept_vals, dtype=float)

    n = len(values)
    if n < 5 or np.std(values) == 0:
        out = {
            "n": int(n),
            "I": np.nan,
            "EI": np.nan,
            "VI_norm": np.nan,
            "SE_norm": np.nan,
            "z_norm": np.nan,
            "p_norm_2s": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
        }
        if permutations and permutations > 0:
            out.update({"p_sim": np.nan, "z_sim": np.nan})
        return out

    # stable ordering
    order = np.argsort(codes)
    codes = [codes[i] for i in order]
    values = values[order]

    w = subset_weights_dense(codes)
    m = Moran(values, w, permutations=permutations)

    I = float(m.I)
    EI = float(m.EI)
    VI = float(m.VI_norm) if np.isfinite(m.VI_norm) else np.nan
    SE = float(np.sqrt(VI)) if np.isfinite(VI) and VI >= 0 else np.nan

    # PySAL also exposes z_norm/p_norm; we compute p ourselves for transparency
    z_norm = float(m.z_norm) if hasattr(m, "z_norm") and np.isfinite(m.z_norm) else (
        (I - EI) / SE if np.isfinite(EI) and np.isfinite(SE) and SE > 0 else np.nan
    )
    p_norm = float(2 * (1 - norm.cdf(abs(z_norm)))) if np.isfinite(z_norm) else np.nan

    zcrit = norm.ppf(1 - alpha/2)
    ci_low = float(I - zcrit * SE) if np.isfinite(SE) else np.nan
    ci_high = float(I + zcrit * SE) if np.isfinite(SE) else np.nan

    out = {
        "n": int(n),
        "I": I,
        "EI": EI,
        "VI_norm": VI,
        "SE_norm": SE,
        "z_norm": z_norm,
        "p_norm_2s": p_norm,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
    }

    if permutations and permutations > 0:
        out.update({"p_sim": float(m.p_sim), "z_sim": float(m.z_sim)})

    return out

# ----------------------------
# Run
# ----------------------------
rows = []
for y in range(START_YEAR, END_YEAR + 1):
    fp = os.path.join(BASE, f"{PREFIX}_{y}.parquet")
    if not os.path.exists(fp):
        rows.append({"year": y, "status": "missing_file"})
        continue

    exp_df, imp_df = agg_year_trade(fp)

    exp_vals = log10_1p(exp_df["exports"].to_numpy())
    imp_vals = log10_1p(imp_df["imports"].to_numpy())

    exp_stats = moran_normal_stats(exp_df["ISO_A3"].tolist(), exp_vals, alpha=ALPHA, permutations=PERMUTATIONS)
    imp_stats = moran_normal_stats(imp_df["ISO_A3"].tolist(), imp_vals, alpha=ALPHA, permutations=PERMUTATIONS)

    row = {
        "year": y,
        "status": "ok",

        # exports
        "n_exports": exp_stats["n"],
        "I_exports": exp_stats["I"],
        "EI_exports": exp_stats["EI"],
        "VI_norm_exports": exp_stats["VI_norm"],
        "SE_norm_exports": exp_stats["SE_norm"],
        "z_norm_exports": exp_stats["z_norm"],
        "p_norm_2s_exports": exp_stats["p_norm_2s"],
        "I_exports_ci95_norm_low": exp_stats["ci95_low"],
        "I_exports_ci95_norm_high": exp_stats["ci95_high"],

        # imports
        "n_imports": imp_stats["n"],
        "I_imports": imp_stats["I"],
        "EI_imports": imp_stats["EI"],
        "VI_norm_imports": imp_stats["VI_norm"],
        "SE_norm_imports": imp_stats["SE_norm"],
        "z_norm_imports": imp_stats["z_norm"],
        "p_norm_2s_imports": imp_stats["p_norm_2s"],
        "I_imports_ci95_norm_low": imp_stats["ci95_low"],
        "I_imports_ci95_norm_high": imp_stats["ci95_high"],
    }

    if PERMUTATIONS and PERMUTATIONS > 0:
        row.update({
            "p_sim_exports": exp_stats["p_sim"],
            "z_sim_exports": exp_stats["z_sim"],
            "p_sim_imports": imp_stats["p_sim"],
            "z_sim_imports": imp_stats["z_sim"],
        })

    rows.append(row)

out = pd.DataFrame(rows).sort_values("year")
out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
out.head(10)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/moran_S2_1976_2023_normal_inference.csv


,year,status,n_exports,I_exports,EI_exports,VI_norm_exports,SE_norm_exports,z_norm_exports,p_norm_2s_exports,I_exports_ci95_norm_low,I_exports_ci95_norm_high,n_imports,I_imports,EI_imports,VI_norm_imports,SE_norm_imports,z_norm_imports,p_norm_2s_imports,I_imports_ci95_norm_low,I_imports_ci95_norm_high
0,1976,ok,140,0.026040,-0.007194,0.000177,0.013313,2.496439,1.254472e-02,-0.000052,0.052132,140,0.011472,-0.007194,0.000177,0.013313,1.402154,1.608693e-01,-0.014620,0.037564
1,1977,ok,139,0.045367,-0.007246,0.000181,0.013448,3.912297,9.142235e-05,0.019009,0.071726,140,0.026714,-0.007194,0.000177,0.013313,2.547106,1.086203e-02,0.000622,0.052806
2,1978,ok,139,0.028605,-0.007246,0.000181,0.013448,2.665894,7.678393e-03,0.002247,0.054964,139,0.022029,-0.007246,0.000181,0.013448,2.176899,2.948809e-02,-0.004329,0.048387
3,1979,ok,139,0.038417,-0.007246,0.000181,0.013448,3.395499,6.850367e-04,0.012059,0.064776,140,0.028899,-0.007194,0.000177,0.013313,2.711251,6.702995e-03,0.002807,0.054991
4,1980,ok,139,0.051559,-0.007246,0.000181,0.013448,4.372671,1.227354e-05,0.025200,0.077917,139,0.046688,-0.007246,0.000181,0.013448,4.010489,6.059320e-05,0.020330,0.073046
5,1981,ok,140,0.055307,-0.007194,0.000177,0.013313,4.694880,2.667639e-06,0.029214,0.081399,140,0.059184,-0.007194,0.000177,0.013313,4.986137,6.159859e-07,0.033092,0.085276
6,1982,ok,140,0.077991,-0.007194,0.000177,0.013313,6.398849,1.565528e-10,0.051899,0.104083,140,0.074337,-0.007194,0.000177,0.013313,6.124365,9.104619e-10,0.048245,0.100429
7,1983,ok,140,0.079484,-0.007194,0.000177,0.013313,6.511011,7.464673e-11,0.053392,0.105576,140,0.062949,-0.007194,0.000177,0.013313,5.268985,1.371801e-07,0.036857,0.089041
8,1984,ok,139,0.085088,-0.007246,0.000181,0.013448,6.865893,6.607603e-12,0.058730,0.111447,139,0.076542,-0.007246,0.000181,0.013448,6.230363,4.653551e-10,0.050183,0.102900
9,1985,ok,139,0.105134,-0.007246,0.000181,0.013448,8.356470,0.000000e+00,0.078776,0.131492,140,0.081634,-0.007194,0.000177,0.013313,6.672533,2.514255e-11,0.055542,0.107726


Sector Estimation - Data Validation previous the the sector based analsyis


In [ ]:
import duckdb

YEAR = 2005
fp = f"{BASE}/S2_{YEAR}.parquet"

con = duckdb.connect()

con.execute(f"""
    SELECT
        typeof(commoditycode) AS type,
        COUNT(*) AS n
    FROM read_parquet('{fp}')
    GROUP BY 1
""").df()


,type,n
0,VARCHAR,2947322


In [ ]:
con.execute(f"""
    SELECT
        LENGTH(CAST(commoditycode AS VARCHAR)) AS len,
        COUNT(*) AS n
    FROM read_parquet('{fp}')
    GROUP BY 1
    ORDER BY 1
""").df()


,len,n
0,4,2947322


In [ ]:
con.execute(f"""
    SELECT DISTINCT
        CAST(commoditycode AS VARCHAR) AS commoditycode
    FROM read_parquet('{fp}')
    ORDER BY commoditycode
    LIMIT 30
""").df()


,commoditycode
0,0011
1,0012
2,0013
3,0014
4,0015
5,0019
6,0111
7,0112
8,0113
9,0114


In [ ]:
con.execute(f"""
    SELECT DISTINCT
        CAST(commoditycode AS VARCHAR) AS commoditycode
    FROM read_parquet('{fp}')
    ORDER BY commoditycode DESC
    LIMIT 30
""").df()


,commoditycode
0,XXXX
1,9710
2,9610
3,9510
4,9410
5,9310
6,9110
7,8999
8,8998
9,8997


In [ ]:
con.execute(f"""
    SELECT
        SUBSTR(LPAD(CAST(commoditycode AS VARCHAR), 2, '0'), 1, 2) AS sitc2,
        COUNT(*) AS n_obs,
        COUNT(DISTINCT exporter) AS n_exporters,
        COUNT(DISTINCT importer) AS n_importers
    FROM read_parquet('{fp}')
    GROUP BY 1
    ORDER BY 1
""").df()


,sitc2,n_obs,n_exporters,n_importers
0,00,7205,198,213
1,01,22074,201,225
2,02,18765,205,225
3,03,33769,225,226
4,04,36529,208,229
...,...,...,...,...
65,94,3454,184,195
66,95,2741,137,206
67,96,745,88,148
68,97,2902,186,194


In [ ]:
con.execute(f"""
    SELECT
        SUBSTR(LPAD(CAST(commoditycode AS VARCHAR), 2, '0'), 1, 2) AS sitc2,
        COUNT(*) AS n_obs,
        COUNT(DISTINCT exporter) AS n_exporters,
        COUNT(DISTINCT importer) AS n_importers
    FROM read_parquet('{fp}')
    GROUP BY 1
    ORDER BY 1
""").df()


,sitc2,n_obs,n_exporters,n_importers
0,00,7205,198,213
1,01,22074,201,225
2,02,18765,205,225
3,03,33769,225,226
4,04,36529,208,229
...,...,...,...,...
65,94,3454,184,195
66,95,2741,137,206
67,96,745,88,148
68,97,2902,186,194


In [ ]:
con.execute(f"""
    WITH base AS (
        SELECT
            UPPER(exporter) AS exporter,
            SUBSTR(LPAD(CAST(commoditycode AS VARCHAR), 2, '0'), 1, 2) AS sitc2,
            value_exporter
        FROM read_parquet('{fp}')
    )
    SELECT
        CASE
            WHEN sitc2 IN ('27','28','32','33','34','35') THEN 'Energy_Minerals'
            WHEN sitc2 IN ('00','01','04','05','22')     THEN 'Bulk_Agriculture'
            WHEN sitc2 IN ('02','03','06','07','11','12') THEN 'Processed_Food'
            WHEN sitc2 IN ('65','84')                   THEN 'Textiles_Apparel'
            WHEN sitc2 IN ('67','68','69')               THEN 'Basic_Metals'
            ELSE 'Other'
        END AS sector_group,
        COUNT(DISTINCT exporter) AS n_countries,
        SUM(value_exporter) AS total_exports
    FROM base
    GROUP BY 1
    ORDER BY n_countries DESC
""").df()


,sector_group,n_countries,total_exports
0,Processed_Food,230,2.807767e+11
1,Basic_Metals,230,7.334934e+11
2,Other,230,6.821088e+12
3,Textiles_Apparel,230,5.079462e+11
4,Energy_Minerals,227,1.238202e+12
5,Bulk_Agriculture,226,2.906499e+11


In [ ]:
import duckdb

YEAR = 2005
fp = f"{BASE}/S2_{YEAR}.parquet"

con = duckdb.connect()

con.execute(f"""
    SELECT
        SUBSTR(commoditycode, 1, 3) AS sitc3,
        COUNT(*) AS n_obs,
        COUNT(DISTINCT exporter) AS n_exporters,
        COUNT(DISTINCT importer) AS n_importers,
        SUM(value_exporter) AS total_exports
    FROM read_parquet('{fp}')
    WHERE SUBSTR(commoditycode, 1, 2) IN ('75','76','77')
    GROUP BY 1
    ORDER BY sitc3
""").df()


,sitc3,n_obs,n_exporters,n_importers,total_exports
0,751,16015,218,227,5.447199e+10
1,752,35356,228,226,2.722189e+11
2,759,17909,223,230,1.832793e+11
3,761,8465,209,225,6.735797e+10
4,762,8991,175,221,1.854164e+10
5,763,9695,204,225,5.501612e+10
6,764,37558,220,228,3.097139e+11
7,771,14165,213,223,4.555775e+10
8,772,17332,222,227,1.423508e+11
9,773,11600,212,227,5.062906e+10


In [ ]:
con.execute(f"""
    WITH base AS (
        SELECT
            UPPER(exporter) AS exporter,
            SUBSTR(commoditycode, 1, 2) AS sitc2,
            SUBSTR(commoditycode, 1, 3) AS sitc3,
            value_exporter
        FROM read_parquet('{fp}')
        WHERE commoditycode != 'XXXX'
          AND SUBSTR(commoditycode, 1, 2) NOT IN ('91','93','94','95','96','97')
    )
    SELECT
        CASE
            WHEN sitc2 = '33' THEN 'Oil'
            WHEN sitc2 = '34' THEN 'Natural_Gas'
            WHEN sitc2 IN ('27','28','32','35') THEN 'Other_Minerals'
            WHEN sitc2 IN ('00','01','04','05','22') THEN 'Bulk_Agriculture'
            WHEN sitc2 IN ('02','03','06','07','11','12') THEN 'Processed_Food'
            WHEN sitc2 IN ('65','84') THEN 'Textiles_Apparel'
            WHEN sitc2 IN ('67','68','69') THEN 'Basic_Metals'
            WHEN sitc2 IN ('51','52','53','54','55') THEN 'Chemicals_Pharma'
            WHEN sitc2 IN ('71','72','73','74') THEN 'General_Machinery'
            WHEN sitc2 IN ('75','76') OR sitc3 = '776'
                THEN 'High_Tech_Electronics'
            WHEN sitc2 = '77' AND sitc3 != '776'
                THEN 'Lower_Tech_Electrical'
            WHEN sitc2 IN ('78','79') THEN 'Transport_Equipment'
            ELSE 'Other'
        END AS sector_group,
        COUNT(DISTINCT exporter) AS n_countries,
        SUM(value_exporter) AS total_exports
    FROM base
    GROUP BY 1
    ORDER BY n_countries DESC
""").df()


,sector_group,n_countries,total_exports
0,Processed_Food,230,2.807767e+11
1,Basic_Metals,230,7.334934e+11
2,Other,230,2.016969e+12
3,Textiles_Apparel,230,5.079462e+11
4,General_Machinery,230,9.320691e+11
5,Lower_Tech_Electrical,230,4.973520e+11
6,Transport_Equipment,229,1.082061e+12
7,Chemicals_Pharma,229,6.629387e+11
8,High_Tech_Electronics,229,1.200162e+12
9,Bulk_Agriculture,226,2.906499e+11


Sectorial Analysisi

In [ ]:
# ============================================================
# Moran's I by year AND sector group with NORMAL-ASSUMPTION inference
# Includes countries with zero sector trade (zeros kept; log10(x+1) used)
#
# Sector groups:
# 1 Oil (SITC2=33)
# 2 Natural Gas (SITC2=34)
# 3 Other Minerals (SITC2 in 27,28,32,35)
# 4 Bulk Agriculture (00,01,04,05,22)
# 5 Processed Food & Beverages (02,03,06,07,11,12)
# 6 Textiles & Apparel (65,84)
# 7 Basic Metals & Metal Products (67,68,69)
# 8 Chemicals & Pharmaceuticals (51,52,53,54,55)
# 9 General Machinery & Industrial Equipment (71,72,73,74)
# 10a High-Tech Electronics & Microelectronics (SITC2 in 75,76 OR SITC3=776)
# 10b Lower-Tech Electrical Machinery (SITC2=77 AND SITC3!=776)
# 11 Transport Equipment (78,79)
#
# IMPORTANT: commoditycode is 4-char SITC; SITC2=first 2 chars; SITC3=first 3 chars
#
# Exclusions:
#   commoditycode='XXXX'
#   SITC2 in {91,93,94,95,96,97}  (special/residual categories)
#
# Outputs per year x sector_group:
#   n, I, EI, VI_norm, SE_norm, z_norm, p_norm_2s, CI95_norm_low/high
# for exports and imports, matching your original outputs.
#
# Optional permutation p_sim/z_sim if PERMUTATIONS>0 (OFF by default)
#
# Stand-alone: reads OD_Matrix.csv + {PREFIX}_YYYY.parquet
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install duckdb libpysal esda scipy

import os
import numpy as np
import pandas as pd
import duckdb

from esda.moran import Moran
from libpysal.weights import W as psW
from scipy.stats import norm

# ----------------------------
# Config (EDIT THESE)
# ----------------------------
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
OD_PATH = os.path.join(BASE, "OD_Matrix.csv")

PREFIX = "S2"          # e.g., "S2"
START_YEAR = 1976
END_YEAR   = 2023

ALPHA = 0.05           # 95% CI
PERMUTATIONS = 0       # set to 999 if you ALSO want p_sim/z_sim saved (optional)

EXPORTER_COL = "exporter"
IMPORTER_COL = "importer"
COMMODITY_COL = "commoditycode"
VALUE_EXPORT_COL = "value_exporter"
VALUE_IMPORT_COL = "value_importer"

OUT_CSV = os.path.join(BASE, f"moran_{PREFIX}_{START_YEAR}_{END_YEAR}_sectors_final_normal_inference.csv")

# ----------------------------
# Sector groups (final taxonomy)
# ----------------------------
SECTOR_ORDER = [
    "Oil",
    "Natural_Gas",
    "Other_Minerals",
    "Bulk_Agriculture",
    "Processed_Food",
    "Textiles_Apparel",
    "Basic_Metals",
    "Chemicals_Pharma",
    "General_Machinery",
    "High_Tech_Electronics",
    "Lower_Tech_Electrical",
    "Transport_Equipment",
]

# Exclusions based on your checks
EXCLUDE_COMMODITYCODE = {"XXXX"}
EXCLUDE_SITC2 = {"91", "93", "94", "95", "96", "97", "XX"}  # 'XX' is a defensive guard

# ----------------------------
# Load OD once -> W_full once (inverse-distance, row-standardized)
# ----------------------------
od_long = pd.read_csv(OD_PATH)
od_long["origin"] = od_long["origin"].astype(str).str.upper()
od_long["destination"] = od_long["destination"].astype(str).str.upper()

D_full = od_long.pivot(index="origin", columns="destination", values="distance_km")
all_codes = sorted(set(D_full.index) | set(D_full.columns))
D_full = D_full.reindex(index=all_codes, columns=all_codes)

np.fill_diagonal(D_full.values, 0.0)

D = D_full.to_numpy(dtype=float)
np.fill_diagonal(D, np.nan)
W_full = 1.0 / D
W_full = W_full / np.nansum(W_full, axis=1, keepdims=True)

code_to_ix = {c: i for i, c in enumerate(all_codes)}
universe_codes = all_codes[:]  # OD universe, fixed across years and sectors

# ----------------------------
# Helpers
# ----------------------------
def log10_1p(x):
    return np.log10(np.asarray(x, dtype=float) + 1.0)

def subset_weights_dense(codes):
    idx = np.array([code_to_ix[c] for c in codes], dtype=int)
    W = W_full[np.ix_(idx, idx)].copy()
    np.fill_diagonal(W, 0.0)

    rs = W.sum(axis=1, keepdims=True)
    rs[rs == 0] = np.nan
    W = W / rs

    neighbors, weights = {}, {}
    for i, c in enumerate(codes):
        js = np.where(np.isfinite(W[i]) & (W[i] > 0))[0]
        neighbors[c] = [codes[j] for j in js]
        weights[c] = [float(W[i, j]) for j in js]

    w = psW(neighbors, weights)
    w.transform = "R"
    return w

def moran_normal_stats(codes, values, alpha=0.05, permutations=0):
    """
    Normal-assumption inference stats for Moran's I.
    Keeps zeros. Drops only non-finite values.
    """
    codes = [str(c).upper() for c in codes]
    values = np.asarray(values, dtype=float)

    # drop only non-finite
    mask = np.isfinite(values)
    codes = [c for c, ok in zip(codes, mask) if ok]
    values = values[mask]

    # keep only OD universe (guard)
    kept_codes, kept_vals = [], []
    for c, v in zip(codes, values):
        if c in code_to_ix:
            kept_codes.append(c)
            kept_vals.append(v)
    codes = kept_codes
    values = np.asarray(kept_vals, dtype=float)

    n = len(values)
    if n < 5 or np.std(values) == 0:
        out = {
            "n": int(n),
            "I": np.nan,
            "EI": np.nan,
            "VI_norm": np.nan,
            "SE_norm": np.nan,
            "z_norm": np.nan,
            "p_norm_2s": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
        }
        if permutations and permutations > 0:
            out.update({"p_sim": np.nan, "z_sim": np.nan})
        return out

    # stable ordering
    order = np.argsort(codes)
    codes = [codes[i] for i in order]
    values = values[order]

    w = subset_weights_dense(codes)
    m = Moran(values, w, permutations=permutations)

    I = float(m.I)
    EI = float(m.EI)
    VI = float(m.VI_norm) if np.isfinite(m.VI_norm) else np.nan
    SE = float(np.sqrt(VI)) if np.isfinite(VI) and VI >= 0 else np.nan

    z_norm = float(m.z_norm) if hasattr(m, "z_norm") and np.isfinite(m.z_norm) else (
        (I - EI) / SE if np.isfinite(EI) and np.isfinite(SE) and SE > 0 else np.nan
    )
    p_norm = float(2 * (1 - norm.cdf(abs(z_norm)))) if np.isfinite(z_norm) else np.nan

    zcrit = norm.ppf(1 - alpha / 2)
    ci_low = float(I - zcrit * SE) if np.isfinite(SE) else np.nan
    ci_high = float(I + zcrit * SE) if np.isfinite(SE) else np.nan

    out = {
        "n": int(n),
        "I": I,
        "EI": EI,
        "VI_norm": VI,
        "SE_norm": SE,
        "z_norm": z_norm,
        "p_norm_2s": p_norm,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
    }

    if permutations and permutations > 0:
        out.update({"p_sim": float(m.p_sim), "z_sim": float(m.z_sim)})

    return out

def agg_year_trade_by_sector(parquet_path: str):
    """
    Returns two dataframes:
      exports: ISO_A3, sector_group, exports
      imports: ISO_A3, sector_group, imports
    Excludes commoditycode='XXXX' and sitc2 in EXCLUDE_SITC2.
    Drops residual 'Other' by leaving unmapped codes as NULL.
    """
    con = duckdb.connect()

    # CASE statement implementing your final taxonomy (2-digit + 3-digit override)
    case_stmt = """
        CASE
            WHEN sitc2 = '33' THEN 'Oil'
            WHEN sitc2 = '34' THEN 'Natural_Gas'
            WHEN sitc2 IN ('27','28','32','35') THEN 'Other_Minerals'
            WHEN sitc2 IN ('00','01','04','05','22') THEN 'Bulk_Agriculture'
            WHEN sitc2 IN ('02','03','06','07','11','12') THEN 'Processed_Food'
            WHEN sitc2 IN ('65','84') THEN 'Textiles_Apparel'
            WHEN sitc2 IN ('67','68','69') THEN 'Basic_Metals'
            WHEN sitc2 IN ('51','52','53','54','55') THEN 'Chemicals_Pharma'
            WHEN sitc2 IN ('71','72','73','74') THEN 'General_Machinery'
            WHEN sitc2 IN ('75','76') OR sitc3 = '776' THEN 'High_Tech_Electronics'
            WHEN sitc2 = '77' AND sitc3 <> '776' THEN 'Lower_Tech_Electrical'
            WHEN sitc2 IN ('78','79') THEN 'Transport_Equipment'
            ELSE NULL
        END
    """

    excl_cc = ",".join([f"'{x}'" for x in EXCLUDE_COMMODITYCODE])
    excl_s2 = ",".join([f"'{x}'" for x in EXCLUDE_SITC2])

    exports = con.execute(f"""
        WITH base AS (
            SELECT
                UPPER({EXPORTER_COL}) AS ISO_A3,
                SUBSTR({COMMODITY_COL}, 1, 2) AS sitc2,
                SUBSTR({COMMODITY_COL}, 1, 3) AS sitc3,
                {COMMODITY_COL} AS commoditycode,
                {VALUE_EXPORT_COL} AS val
            FROM read_parquet('{parquet_path}')
        ),
        filtered AS (
            SELECT
                ISO_A3,
                {case_stmt} AS sector_group,
                val
            FROM base
            WHERE commoditycode NOT IN ({excl_cc})
              AND sitc2 NOT IN ({excl_s2})
        )
        SELECT
            ISO_A3,
            sector_group,
            SUM(val) AS exports
        FROM filtered
        WHERE sector_group IS NOT NULL
        GROUP BY 1,2
    """).df()

    imports = con.execute(f"""
        WITH base AS (
            SELECT
                UPPER({IMPORTER_COL}) AS ISO_A3,
                SUBSTR({COMMODITY_COL}, 1, 2) AS sitc2,
                SUBSTR({COMMODITY_COL}, 1, 3) AS sitc3,
                {COMMODITY_COL} AS commoditycode,
                {VALUE_IMPORT_COL} AS val
            FROM read_parquet('{parquet_path}')
        ),
        filtered AS (
            SELECT
                ISO_A3,
                {case_stmt} AS sector_group,
                val
            FROM base
            WHERE commoditycode NOT IN ({excl_cc})
              AND sitc2 NOT IN ({excl_s2})
        )
        SELECT
            ISO_A3,
            sector_group,
            SUM(val) AS imports
        FROM filtered
        WHERE sector_group IS NOT NULL
        GROUP BY 1,2
    """).df()

    con.close()
    return exports, imports

def build_full_vector(df, value_col, sector_group, universe_codes):
    """
    df has columns ISO_A3, sector_group, value_col.
    Returns aligned values array for universe_codes, with missing filled as 0.
    """
    if df is None or len(df) == 0:
        return np.zeros(len(universe_codes), dtype=float)

    sub = df[df["sector_group"] == sector_group][["ISO_A3", value_col]].copy()
    sub["ISO_A3"] = sub["ISO_A3"].astype(str).str.upper()

    s = pd.Series(0.0, index=universe_codes, dtype=float)
    if len(sub) > 0:
        g = sub.groupby("ISO_A3")[value_col].sum()
        g = g[g.index.isin(s.index)]
        s.loc[g.index] = g.values

    return s.values

# ----------------------------
# Run
# ----------------------------
rows = []

for y in range(START_YEAR, END_YEAR + 1):
    fp = os.path.join(BASE, f"{PREFIX}_{y}.parquet")
    if not os.path.exists(fp):
        for sector in SECTOR_ORDER:
            rows.append({"year": y, "sector_group": sector, "status": "missing_file"})
        continue

    exp_sec_df, imp_sec_df = agg_year_trade_by_sector(fp)

    for sector in SECTOR_ORDER:
        # Full universe vectors with zeros for missing country-sector values
        exp_raw = build_full_vector(exp_sec_df, "exports", sector, universe_codes)
        imp_raw = build_full_vector(imp_sec_df, "imports", sector, universe_codes)

        # log10(x+1)
        exp_vals = log10_1p(exp_raw)
        imp_vals = log10_1p(imp_raw)

        exp_stats = moran_normal_stats(universe_codes, exp_vals, alpha=ALPHA, permutations=PERMUTATIONS)
        imp_stats = moran_normal_stats(universe_codes, imp_vals, alpha=ALPHA, permutations=PERMUTATIONS)

        row = {
            "year": y,
            "sector_group": sector,
            "status": "ok",

            # exports
            "n_exports": exp_stats["n"],
            "I_exports": exp_stats["I"],
            "EI_exports": exp_stats["EI"],
            "VI_norm_exports": exp_stats["VI_norm"],
            "SE_norm_exports": exp_stats["SE_norm"],
            "z_norm_exports": exp_stats["z_norm"],
            "p_norm_2s_exports": exp_stats["p_norm_2s"],
            "I_exports_ci95_norm_low": exp_stats["ci95_low"],
            "I_exports_ci95_norm_high": exp_stats["ci95_high"],

            # imports
            "n_imports": imp_stats["n"],
            "I_imports": imp_stats["I"],
            "EI_imports": imp_stats["EI"],
            "VI_norm_imports": imp_stats["VI_norm"],
            "SE_norm_imports": imp_stats["SE_norm"],
            "z_norm_imports": imp_stats["z_norm"],
            "p_norm_2s_imports": imp_stats["p_norm_2s"],
            "I_imports_ci95_norm_low": imp_stats["ci95_low"],
            "I_imports_ci95_norm_high": imp_stats["ci95_high"],
        }

        if PERMUTATIONS and PERMUTATIONS > 0:
            row.update({
                "p_sim_exports": exp_stats["p_sim"],
                "z_sim_exports": exp_stats["z_sim"],
                "p_sim_imports": imp_stats["p_sim"],
                "z_sim_imports": imp_stats["z_sim"],
            })

        rows.append(row)

out = pd.DataFrame(rows).sort_values(["year", "sector_group"])
out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
out.head(20)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/moran_S2_1976_2023_sectors_final_normal_inference.csv


,year,sector_group,status,n_exports,I_exports,EI_exports,VI_norm_exports,SE_norm_exports,z_norm_exports,p_norm_2s_exports,...,I_exports_ci95_norm_high,n_imports,I_imports,EI_imports,VI_norm_imports,SE_norm_imports,z_norm_imports,p_norm_2s_imports,I_imports_ci95_norm_low,I_imports_ci95_norm_high
6,1976,Basic_Metals,ok,171,0.046708,-0.005882,0.000122,0.011057,4.756157,1.973126e-06,...,0.068380,171,0.045805,-0.005882,0.000122,0.011057,4.674454,2.947372e-06,0.024133,0.067477
3,1976,Bulk_Agriculture,ok,171,0.063910,-0.005882,0.000122,0.011057,6.311898,2.756344e-10,...,0.085582,171,0.047655,-0.005882,0.000122,0.011057,4.841794,1.286724e-06,0.025983,0.069327
7,1976,Chemicals_Pharma,ok,171,0.061586,-0.005882,0.000122,0.011057,6.101705,1.049430e-09,...,0.083258,171,0.047332,-0.005882,0.000122,0.011057,4.812603,1.489773e-06,0.025660,0.069004
8,1976,General_Machinery,ok,171,0.044230,-0.005882,0.000122,0.011057,4.532095,5.840147e-06,...,0.065902,171,0.046497,-0.005882,0.000122,0.011057,4.737058,2.168436e-06,0.024825,0.068169
9,1976,High_Tech_Electronics,ok,171,0.043972,-0.005882,0.000122,0.011057,4.508750,6.521071e-06,...,0.065644,171,0.044354,-0.005882,0.000122,0.011057,4.543309,5.537790e-06,0.022683,0.066026
10,1976,Lower_Tech_Electrical,ok,171,0.053175,-0.005882,0.000122,0.011057,5.341003,9.243368e-08,...,0.074847,171,0.044755,-0.005882,0.000122,0.011057,4.579498,4.660931e-06,0.023083,0.066427
1,1976,Natural_Gas,ok,171,0.019497,-0.005882,0.000122,0.011057,2.295294,2.171627e-02,...,0.041169,171,0.060439,-0.005882,0.000122,0.011057,5.997962,1.998098e-09,0.038767,0.082111
0,1976,Oil,ok,171,0.038898,-0.005882,0.000122,0.011057,4.049878,5.124442e-05,...,0.060570,171,0.049349,-0.005882,0.000122,0.011057,4.995041,5.882332e-07,0.027677,0.071021
2,1976,Other_Minerals,ok,171,0.059206,-0.005882,0.000122,0.011057,5.886461,3.945523e-09,...,0.080878,171,0.042539,-0.005882,0.000122,0.011057,4.379157,1.191393e-05,0.020867,0.064211
4,1976,Processed_Food,ok,171,0.076575,-0.005882,0.000122,0.011057,7.457241,8.837375e-14,...,0.098247,171,0.044975,-0.005882,0.000122,0.011057,4.599394,4.237218e-06,0.023303,0.066647


Graphs

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------- CONFIG --------
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
CSV_PATH = os.path.join(BASE, "moran_S2_1976_2023_sectors_final_normal_inference.csv")  # adjust if your filename differs

OUT_DIR = os.path.join(BASE, "figures_moran_sectors")
os.makedirs(OUT_DIR, exist_ok=True)

SECTOR_ORDER = [
    "Oil",
    "Natural_Gas",
    "Other_Minerals",
    "Bulk_Agriculture",
    "Processed_Food",
    "Textiles_Apparel",
    "Basic_Metals",
    "Chemicals_Pharma",
    "General_Machinery",
    "High_Tech_Electronics",
    "Lower_Tech_Electrical",
    "Transport_Equipment",
]

# Optional: fixed y-limits for comparability (set after checking your data)
Y_LIM = (-0.05, 0.30)

# -------- LOAD --------
df = pd.read_csv(CSV_PATH)

# Keep only ok rows (leave missing_file out of plotting)
df = df[df["status"] == "ok"].copy()
df["year"] = df["year"].astype(int)

# -------- HELPERS --------
def plot_panel(ax, years, I, lo, hi, title, y_lim=None):
    # CI band
    ax.fill_between(years, lo, hi, alpha=0.25, linewidth=0)

    # Moran I line
    ax.plot(years, I, linewidth=2)

    # zero line
    ax.axhline(0, linestyle="--", linewidth=1, alpha=0.5)

    ax.set_title(title)
    ax.set_xlim(years.min(), years.max())
    if y_lim is not None:
        ax.set_ylim(*y_lim)

    # Light formatting
    ax.grid(True, alpha=0.25)
    ax.tick_params(axis="x", labelrotation=90)

def plot_sector_2panel(df_sector, sector, out_path, y_lim=None):
    df_sector = df_sector.sort_values("year")

    years = df_sector["year"].to_numpy()

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    # Exports
    plot_panel(
        axes[0],
        years,
        df_sector["I_exports"].to_numpy(),
        df_sector["I_exports_ci95_norm_low"].to_numpy(),
        df_sector["I_exports_ci95_norm_high"].to_numpy(),
        title="Moran I – Exports",
        y_lim=y_lim
    )

    # Imports
    plot_panel(
        axes[1],
        years,
        df_sector["I_imports"].to_numpy(),
        df_sector["I_imports_ci95_norm_low"].to_numpy(),
        df_sector["I_imports_ci95_norm_high"].to_numpy(),
        title="Moran I – Imports",
        y_lim=y_lim
    )

    fig.suptitle(f"Moran I – {sector}", y=0.98)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def master_figure(df, sectors, mode, out_path, ncols=3, y_lim=None):
    """
    mode: 'exports' or 'imports'
    Creates a grid of small multiples: one panel per sector
    """
    assert mode in ("exports", "imports")

    n = len(sectors)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.2*nrows), sharex=False, sharey=True)
    axes = np.array(axes).reshape(nrows, ncols)

    for i, sector in enumerate(sectors):
        r, c = divmod(i, ncols)
        ax = axes[r, c]

        d = df[df["sector_group"] == sector].sort_values("year")
        if d.empty:
            ax.set_title(f"{sector} (no data)")
            ax.axis("off")
            continue

        years = d["year"].to_numpy()

        if mode == "exports":
            I = d["I_exports"].to_numpy()
            lo = d["I_exports_ci95_norm_low"].to_numpy()
            hi = d["I_exports_ci95_norm_high"].to_numpy()
        else:
            I = d["I_imports"].to_numpy()
            lo = d["I_imports_ci95_norm_low"].to_numpy()
            hi = d["I_imports_ci95_norm_high"].to_numpy()

        ax.fill_between(years, lo, hi, alpha=0.25, linewidth=0)
        ax.plot(years, I, linewidth=1.8)
        ax.axhline(0, linestyle="--", linewidth=1, alpha=0.4)
        ax.set_title(sector, fontsize=11)

        ax.set_xlim(years.min(), years.max())
        if y_lim is not None:
            ax.set_ylim(*y_lim)

        ax.grid(True, alpha=0.2)

        # fewer x ticks to keep it readable
        xt = years[::5]  # every 5 years
        ax.set_xticks(xt)
        ax.tick_params(axis="x", labelrotation=90, labelsize=8)
        ax.tick_params(axis="y", labelsize=8)

    # Turn off unused axes
    for j in range(n, nrows*ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    fig.suptitle(f"Moran I – {mode.capitalize()} (all sectors)", y=0.995, fontsize=14)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

# -------- 1) One figure per sector (2-panel) --------
for sector in SECTOR_ORDER:
    dsec = df[df["sector_group"] == sector].copy()
    out_path = os.path.join(OUT_DIR, f"moran_{sector}.png")
    plot_sector_2panel(dsec, sector, out_path, y_lim=Y_LIM)

print("Saved per-sector figures in:", OUT_DIR)

# -------- 2) Master figures (all sectors) --------
master_exports = os.path.join(OUT_DIR, "MASTER_moran_all_sectors_exports.png")
master_imports = os.path.join(OUT_DIR, "MASTER_moran_all_sectors_imports.png")

master_figure(df, SECTOR_ORDER, mode="exports", out_path=master_exports, ncols=3, y_lim=Y_LIM)
master_figure(df, SECTOR_ORDER, mode="imports", out_path=master_imports, ncols=3, y_lim=Y_LIM)

print("Saved master figures:", master_exports, "and", master_imports)


Saved per-sector figures in: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/figures_moran_sectors
Saved master figures: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/figures_moran_sectors/MASTER_moran_all_sectors_exports.png and /content/drive/My Drive/96 Colab Notebooks/Growth Lab/figures_moran_sectors/MASTER_moran_all_sectors_imports.png


2 digit SITC Code Analysis

In [ ]:
# ============================================================
# Moran's I by year AND SITC-2 category with NORMAL-ASSUMPTION inference
# Includes countries with zero SITC-2 trade (zeros kept; log10(x+1) used)
#
# SITC-2 universe = the list you provided (including 89), all treated at 2 digits.
# commoditycode is 4-char SITC (VARCHAR); SITC2 = first 2 chars.
#
# Exclusions:
#   commoditycode='XXXX'
#   SITC2 in {91,93,94,95,96,97,XX}  (special/residual)
#
# Outputs per year x sitc2:
#   n, I, EI, VI_norm, SE_norm, z_norm, p_norm_2s, CI95_norm_low/high
# for exports and imports, matching your original outputs.
#
# Optional permutation p_sim/z_sim if PERMUTATIONS>0 (OFF by default)
#
# Stand-alone: reads OD_Matrix.csv + {PREFIX}_YYYY.parquet
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install duckdb libpysal esda scipy

import os
import numpy as np
import pandas as pd
import duckdb

from esda.moran import Moran
from libpysal.weights import W as psW
from scipy.stats import norm

# ----------------------------
# Config (EDIT THESE)
# ----------------------------
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
OD_PATH = os.path.join(BASE, "OD_Matrix.csv")

PREFIX = "S2"
START_YEAR = 1976
END_YEAR   = 2023

ALPHA = 0.05           # 95% CI
PERMUTATIONS = 0       # set to 999 if you ALSO want p_sim/z_sim saved (optional)

EXPORTER_COL = "exporter"
IMPORTER_COL = "importer"
COMMODITY_COL = "commoditycode"
VALUE_EXPORT_COL = "value_exporter"
VALUE_IMPORT_COL = "value_importer"

OUT_CSV = os.path.join(BASE, f"moran_{PREFIX}_{START_YEAR}_{END_YEAR}_sitc2_normal_inference.csv")

# ----------------------------
# SITC-2 universe (as provided)
# ----------------------------
SITC2_LIST = [
    "00","01","02","03","04","05","06","07","08","09","11","12",
    "21","22","23","24","25","26","27","28","29",
    "32","33","34","35",
    "41","42","43",
    "51","52","53","54","55","56","57","58","59",
    "61","62","63","64","65","66","67","68","69",
    "71","72","73","74","75","76","77","78","79",
    "81","82","83","84","85","87","88","89"
]

# Exclusions based on your checks
EXCLUDE_COMMODITYCODE = {"XXXX"}
EXCLUDE_SITC2 = {"91","93","94","95","96","97","XX"}  # defensive guard

# ----------------------------
# Load OD once -> W_full once (inverse-distance, row-standardized)
# ----------------------------
od_long = pd.read_csv(OD_PATH)
od_long["origin"] = od_long["origin"].astype(str).str.upper()
od_long["destination"] = od_long["destination"].astype(str).str.upper()

D_full = od_long.pivot(index="origin", columns="destination", values="distance_km")
all_codes = sorted(set(D_full.index) | set(D_full.columns))
D_full = D_full.reindex(index=all_codes, columns=all_codes)

np.fill_diagonal(D_full.values, 0.0)

D = D_full.to_numpy(dtype=float)
np.fill_diagonal(D, np.nan)
W_full = 1.0 / D
W_full = W_full / np.nansum(W_full, axis=1, keepdims=True)

code_to_ix = {c: i for i, c in enumerate(all_codes)}
universe_codes = all_codes[:]  # OD universe, fixed across years and SITC2

# ----------------------------
# Helpers
# ----------------------------
def log10_1p(x):
    return np.log10(np.asarray(x, dtype=float) + 1.0)

def subset_weights_dense(codes):
    idx = np.array([code_to_ix[c] for c in codes], dtype=int)
    W = W_full[np.ix_(idx, idx)].copy()
    np.fill_diagonal(W, 0.0)

    rs = W.sum(axis=1, keepdims=True)
    rs[rs == 0] = np.nan
    W = W / rs

    neighbors, weights = {}, {}
    for i, c in enumerate(codes):
        js = np.where(np.isfinite(W[i]) & (W[i] > 0))[0]
        neighbors[c] = [codes[j] for j in js]
        weights[c] = [float(W[i, j]) for j in js]

    w = psW(neighbors, weights)
    w.transform = "R"
    return w

def moran_normal_stats(codes, values, alpha=0.05, permutations=0):
    """
    Normal-assumption inference stats for Moran's I.
    Keeps zeros. Drops only non-finite values.
    """
    codes = [str(c).upper() for c in codes]
    values = np.asarray(values, dtype=float)

    # drop only non-finite
    mask = np.isfinite(values)
    codes = [c for c, ok in zip(codes, mask) if ok]
    values = values[mask]

    # keep only OD universe (guard)
    kept_codes, kept_vals = [], []
    for c, v in zip(codes, values):
        if c in code_to_ix:
            kept_codes.append(c)
            kept_vals.append(v)
    codes = kept_codes
    values = np.asarray(kept_vals, dtype=float)

    n = len(values)
    if n < 5 or np.std(values) == 0:
        out = {
            "n": int(n),
            "I": np.nan,
            "EI": np.nan,
            "VI_norm": np.nan,
            "SE_norm": np.nan,
            "z_norm": np.nan,
            "p_norm_2s": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
        }
        if permutations and permutations > 0:
            out.update({"p_sim": np.nan, "z_sim": np.nan})
        return out

    # stable ordering
    order = np.argsort(codes)
    codes = [codes[i] for i in order]
    values = values[order]

    w = subset_weights_dense(codes)
    m = Moran(values, w, permutations=permutations)

    I = float(m.I)
    EI = float(m.EI)
    VI = float(m.VI_norm) if np.isfinite(m.VI_norm) else np.nan
    SE = float(np.sqrt(VI)) if np.isfinite(VI) and VI >= 0 else np.nan

    z_norm = float(m.z_norm) if hasattr(m, "z_norm") and np.isfinite(m.z_norm) else (
        (I - EI) / SE if np.isfinite(EI) and np.isfinite(SE) and SE > 0 else np.nan
    )
    p_norm = float(2 * (1 - norm.cdf(abs(z_norm)))) if np.isfinite(z_norm) else np.nan

    zcrit = norm.ppf(1 - alpha/2)
    ci_low = float(I - zcrit * SE) if np.isfinite(SE) else np.nan
    ci_high = float(I + zcrit * SE) if np.isfinite(SE) else np.nan

    out = {
        "n": int(n),
        "I": I,
        "EI": EI,
        "VI_norm": VI,
        "SE_norm": SE,
        "z_norm": z_norm,
        "p_norm_2s": p_norm,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
    }

    if permutations and permutations > 0:
        out.update({"p_sim": float(m.p_sim), "z_sim": float(m.z_sim)})

    return out

def agg_year_trade_by_sitc2(parquet_path: str, sitc2_list):
    """
    Returns two dataframes:
      exports: ISO_A3, sitc2, exports
      imports: ISO_A3, sitc2, imports
    Only includes sitc2 in sitc2_list (whitelist).
    Excludes commoditycode='XXXX' and sitc2 in EXCLUDE_SITC2.
    """
    con = duckdb.connect()

    excl_cc = ",".join([f"'{x}'" for x in EXCLUDE_COMMODITYCODE])
    excl_s2 = ",".join([f"'{x}'" for x in EXCLUDE_SITC2])
    s2_list = ",".join([f"'{x}'" for x in sitc2_list])

    exports = con.execute(f"""
        WITH base AS (
            SELECT
                UPPER({EXPORTER_COL}) AS ISO_A3,
                SUBSTR({COMMODITY_COL}, 1, 2) AS sitc2,
                {COMMODITY_COL} AS commoditycode,
                {VALUE_EXPORT_COL} AS val
            FROM read_parquet('{parquet_path}')
        )
        SELECT
            ISO_A3,
            sitc2,
            SUM(val) AS exports
        FROM base
        WHERE commoditycode NOT IN ({excl_cc})
          AND sitc2 NOT IN ({excl_s2})
          AND sitc2 IN ({s2_list})
        GROUP BY 1,2
    """).df()

    imports = con.execute(f"""
        WITH base AS (
            SELECT
                UPPER({IMPORTER_COL}) AS ISO_A3,
                SUBSTR({COMMODITY_COL}, 1, 2) AS sitc2,
                {COMMODITY_COL} AS commoditycode,
                {VALUE_IMPORT_COL} AS val
            FROM read_parquet('{parquet_path}')
        )
        SELECT
            ISO_A3,
            sitc2,
            SUM(val) AS imports
        FROM base
        WHERE commoditycode NOT IN ({excl_cc})
          AND sitc2 NOT IN ({excl_s2})
          AND sitc2 IN ({s2_list})
        GROUP BY 1,2
    """).df()

    con.close()
    return exports, imports

def build_full_vector_sitc2(df, value_col, sitc2, universe_codes):
    """
    df has columns ISO_A3, sitc2, value_col.
    Returns aligned values array for universe_codes, with missing filled as 0.
    """
    if df is None or len(df) == 0:
        return np.zeros(len(universe_codes), dtype=float)

    sub = df[df["sitc2"] == sitc2][["ISO_A3", value_col]].copy()
    sub["ISO_A3"] = sub["ISO_A3"].astype(str).str.upper()

    s = pd.Series(0.0, index=universe_codes, dtype=float)
    if len(sub) > 0:
        g = sub.groupby("ISO_A3")[value_col].sum()
        g = g[g.index.isin(s.index)]
        s.loc[g.index] = g.values

    return s.values

# ----------------------------
# Run
# ----------------------------
rows = []

for y in range(START_YEAR, END_YEAR + 1):
    fp = os.path.join(BASE, f"{PREFIX}_{y}.parquet")
    if not os.path.exists(fp):
        for s2 in SITC2_LIST:
            rows.append({"year": y, "sitc2": s2, "status": "missing_file"})
        continue

    exp_s2_df, imp_s2_df = agg_year_trade_by_sitc2(fp, SITC2_LIST)

    for s2 in SITC2_LIST:
        exp_raw = build_full_vector_sitc2(exp_s2_df, "exports", s2, universe_codes)
        imp_raw = build_full_vector_sitc2(imp_s2_df, "imports", s2, universe_codes)

        exp_vals = log10_1p(exp_raw)
        imp_vals = log10_1p(imp_raw)

        exp_stats = moran_normal_stats(universe_codes, exp_vals, alpha=ALPHA, permutations=PERMUTATIONS)
        imp_stats = moran_normal_stats(universe_codes, imp_vals, alpha=ALPHA, permutations=PERMUTATIONS)

        row = {
            "year": y,
            "sitc2": s2,
            "status": "ok",

            # exports
            "n_exports": exp_stats["n"],
            "I_exports": exp_stats["I"],
            "EI_exports": exp_stats["EI"],
            "VI_norm_exports": exp_stats["VI_norm"],
            "SE_norm_exports": exp_stats["SE_norm"],
            "z_norm_exports": exp_stats["z_norm"],
            "p_norm_2s_exports": exp_stats["p_norm_2s"],
            "I_exports_ci95_norm_low": exp_stats["ci95_low"],
            "I_exports_ci95_norm_high": exp_stats["ci95_high"],

            # imports
            "n_imports": imp_stats["n"],
            "I_imports": imp_stats["I"],
            "EI_imports": imp_stats["EI"],
            "VI_norm_imports": imp_stats["VI_norm"],
            "SE_norm_imports": imp_stats["VI_norm"],
            "SE_norm_imports": imp_stats["SE_norm"],
            "z_norm_imports": imp_stats["z_norm"],
            "p_norm_2s_imports": imp_stats["p_norm_2s"],
            "I_imports_ci95_norm_low": imp_stats["ci95_low"],
            "I_imports_ci95_norm_high": imp_stats["ci95_high"],
        }

        if PERMUTATIONS and PERMUTATIONS > 0:
            row.update({
                "p_sim_exports": exp_stats["p_sim"],
                "z_sim_exports": exp_stats["z_sim"],
                "p_sim_imports": imp_stats["p_sim"],
                "z_sim_imports": imp_stats["z_sim"],
            })

        rows.append(row)

out = pd.DataFrame(rows).sort_values(["year", "sitc2"])
out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
out.head(20)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/moran_S2_1976_2023_sitc2_normal_inference.csv


,year,sitc2,status,n_exports,I_exports,EI_exports,VI_norm_exports,SE_norm_exports,z_norm_exports,p_norm_2s_exports,...,I_exports_ci95_norm_high,n_imports,I_imports,EI_imports,VI_norm_imports,SE_norm_imports,z_norm_imports,p_norm_2s_imports,I_imports_ci95_norm_low,I_imports_ci95_norm_high
0,1976,00,ok,171,0.038525,-0.005882,0.000122,0.011057,4.016126,5.916255e-05,...,0.060197,171,0.043942,-0.005882,0.000122,0.011057,4.505973,6.606939e-06,0.022270,0.065614
1,1976,01,ok,171,0.095744,-0.005882,0.000122,0.011057,9.190832,0.000000e+00,...,0.117416,171,0.048493,-0.005882,0.000122,0.011057,4.917548,8.763501e-07,0.026821,0.070165
2,1976,02,ok,171,0.056165,-0.005882,0.000122,0.011057,5.611382,2.007173e-08,...,0.077836,171,0.046591,-0.005882,0.000122,0.011057,4.745616,2.078730e-06,0.024920,0.068263
3,1976,03,ok,171,0.072228,-0.005882,0.000122,0.011057,7.064164,1.615819e-12,...,0.093900,171,0.053731,-0.005882,0.000122,0.011057,5.391337,6.993521e-08,0.032059,0.075403
4,1976,04,ok,171,0.054714,-0.005882,0.000122,0.011057,5.480193,4.248631e-08,...,0.076386,171,0.048873,-0.005882,0.000122,0.011057,4.951984,7.346053e-07,0.027201,0.070545
5,1976,05,ok,171,0.061397,-0.005882,0.000122,0.011057,6.084557,1.168136e-09,...,0.083069,171,0.049350,-0.005882,0.000122,0.011057,4.995066,5.881574e-07,0.027678,0.071022
6,1976,06,ok,171,0.091484,-0.005882,0.000122,0.011057,8.805590,0.000000e+00,...,0.113156,171,0.043515,-0.005882,0.000122,0.011057,4.467345,7.919651e-06,0.021843,0.065186
7,1976,07,ok,171,0.067285,-0.005882,0.000122,0.011057,6.617057,3.664202e-11,...,0.088957,171,0.042442,-0.005882,0.000122,0.011057,4.370335,1.240559e-05,0.020770,0.064114
8,1976,08,ok,171,0.045360,-0.005882,0.000122,0.011057,4.634266,3.582058e-06,...,0.067032,171,0.053972,-0.005882,0.000122,0.011057,5.413121,6.193560e-08,0.032300,0.075644
9,1976,09,ok,171,0.070679,-0.005882,0.000122,0.011057,6.924060,4.388712e-12,...,0.092351,171,0.049642,-0.005882,0.000122,0.011057,5.021498,5.126993e-07,0.027970,0.071314


Graphs

In [ ]:
import os
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# CONFIG
# -----------------------------
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
CSV_PATH = os.path.join(BASE, "moran_S2_1976_2023_sitc2_normal_inference.csv")  # <-- adjust if needed

OUT_DIR = os.path.join(BASE, "figures_moran_sitc2")
IND_DIR = os.path.join(OUT_DIR, "individual")
GRID_DIR = os.path.join(OUT_DIR, "grids_4x4")

os.makedirs(IND_DIR, exist_ok=True)
os.makedirs(GRID_DIR, exist_ok=True)

# Optional: choose a fixed y-range for all plots (recommended for comparability).
# If set to None, code will compute global limits from the data.
Y_LIM = None  # e.g. (-0.10, 0.35)

# -----------------------------
# SITC-2 labels (2-digit + name)
# -----------------------------
SITC2_NAME = {
    "00": "Live animals other than fish",
    "01": "Meat and meat preparations",
    "02": "Dairy products and birds’ eggs",
    "03": "Fish, crustaceans, molluscs and preparations",
    "04": "Cereals and cereal preparations",
    "05": "Vegetables and fruit",
    "06": "Sugars, sugar preparations and honey",
    "07": "Coffee, tea, cocoa, spices and manufactures",
    "08": "Feeding stuff for animals (excl. unmilled cereals)",
    "09": "Miscellaneous edible products and preparations",
    "11": "Beverages",
    "12": "Tobacco and tobacco manufactures",
    "21": "Hides, skins, and furskins, raw",
    "22": "Oil-seeds and oleaginous fruits; seeds; plants; straw/fodder",
    "23": "Crude rubber (incl. synthetic and reclaimed)",
    "24": "Cork and wood",
    "25": "Pulp and waste paper",
    "26": "Textile fibres and their wastes (other than wool tops)",
    "27": "Crude fertilizers and crude minerals",
    "28": "Metalliferous ores and metal scrap",
    "29": "Crude animal and vegetable materials, n.e.s.",
    "32": "Coal, coke, and briquettes",
    "33": "Petroleum, petroleum products, and related materials",
    "34": "Gas, natural and manufactured",
    "35": "Electric current",
    "41": "Animal oils and fats",
    "42": "Fixed vegetable oils and fats",
    "43": "Animal/vegetable fats and oils; cleavage products; prepared fats",
    "51": "Organic chemicals",
    "52": "Inorganic chemicals",
    "53": "Dyeing, tanning and coloring materials",
    "54": "Medicinal and pharmaceutical products",
    "55": "Essential oils, perfumes, cosmetics",
    "56": "Fertilizers (manufactured)",
    "57": "Plastics in primary forms",
    "58": "Plastics in non-primary forms; plastic articles",
    "59": "Chemical materials and products, n.e.s.",
    "61": "Leather and leather manufactures; dressed furskins",
    "62": "Rubber manufactures, n.e.s.",
    "63": "Cork and wood manufactures (excl. furniture)",
    "64": "Paper, paperboard, and articles thereof",
    "65": "Textile yarn, fabrics, made-up articles, n.e.s.",
    "66": "Non-metallic mineral manufactures, n.e.s.",
    "67": "Iron and steel",
    "68": "Non-ferrous metals",
    "69": "Manufactures of metals, n.e.s.",
    "71": "Power-generating machinery and equipment",
    "72": "Machinery specialized for particular industries",
    "73": "Metalworking machinery",
    "74": "General industrial machinery and equipment",
    "75": "Office machines and ADP equipment",
    "76": "Telecommunications and sound-recording equipment",
    "77": "Electrical machinery, apparatus and appliances, n.e.s.",
    "78": "Road vehicles (incl. air-cushion vehicles)",
    "79": "Other transport equipment",
    "81": "Prefabricated buildings; sanitary/plumbing/heating/lighting fixtures",
    "82": "Furniture and parts; bedding, mattresses, supports",
    "83": "Travel goods, handbags, and similar containers",
    "84": "Articles of apparel and clothing accessories",
    "85": "Footwear",
    "87": "Photographic/optical goods; watches and clocks",
    "88": "Miscellaneous manufactured articles, n.e.s.",
    "89": "Commodities and transactions not classified elsewhere in the SITC",
}

SITC2_LIST = list(SITC2_NAME.keys())  # in the order above

# -----------------------------
# LOAD + CLEAN
# -----------------------------
df = pd.read_csv(CSV_PATH)
df = df[df["status"] == "ok"].copy()
df["year"] = df["year"].astype(int)
df["sitc2"] = df["sitc2"].astype(str).str.zfill(2)

# Keep only codes we have names for (defensive)
df = df[df["sitc2"].isin(SITC2_LIST)].copy()

# -----------------------------
# Y-limits (global, consistent)
# -----------------------------
if Y_LIM is None:
    cols = [
        "I_exports_ci95_norm_low","I_exports_ci95_norm_high",
        "I_imports_ci95_norm_low","I_imports_ci95_norm_high",
        "I_exports","I_imports"
    ]
    vmin = np.nanmin(df[cols].to_numpy())
    vmax = np.nanmax(df[cols].to_numpy())
    pad = 0.05 * (vmax - vmin) if np.isfinite(vmax - vmin) and (vmax - vmin) > 0 else 0.05
    Y_LIM = (float(vmin - pad), float(vmax + pad))

# -----------------------------
# Helpers
# -----------------------------
def safe_filename(s: str) -> str:
    s = s.strip()
    s = re.sub(r"[^\w\-\.\s]", "", s)
    s = re.sub(r"\s+", "_", s)
    return s[:180]

def plot_two_panel_sector(d: pd.DataFrame, sitc2: str, name: str, out_path: str, y_lim):
    d = d.sort_values("year")
    years = d["year"].to_numpy()

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    # Exports
    axes[0].fill_between(
        years,
        d["I_exports_ci95_norm_low"].to_numpy(),
        d["I_exports_ci95_norm_high"].to_numpy(),
        alpha=0.25,
        linewidth=0
    )
    axes[0].plot(years, d["I_exports"].to_numpy(), linewidth=2)
    axes[0].axhline(0, linestyle="--", linewidth=1, alpha=0.5)
    axes[0].set_title("Exports")
    axes[0].set_ylim(*y_lim)
    axes[0].grid(True, alpha=0.25)

    # Imports
    axes[1].fill_between(
        years,
        d["I_imports_ci95_norm_low"].to_numpy(),
        d["I_imports_ci95_norm_high"].to_numpy(),
        alpha=0.25,
        linewidth=0
    )
    axes[1].plot(years, d["I_imports"].to_numpy(), linewidth=2)
    axes[1].axhline(0, linestyle="--", linewidth=1, alpha=0.5)
    axes[1].set_title("Imports")
    axes[1].set_ylim(*y_lim)
    axes[1].grid(True, alpha=0.25)

    axes[1].tick_params(axis="x", labelrotation=90)

    fig.suptitle(f"Moran's I – {sitc2}: {name}", y=0.98)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def plot_grid_4x4(df: pd.DataFrame, sitc2_codes: list, mode: str, out_path: str, y_lim):
    """
    mode in {'exports','imports'}
    One subplot per sitc2: line + CI band.
    """
    assert mode in ("exports", "imports")

    fig, axes = plt.subplots(4, 4, figsize=(20, 14), sharey=True)
    axes = axes.flatten()

    for i in range(16):
        ax = axes[i]
        if i >= len(sitc2_codes):
            ax.axis("off")
            continue

        code = sitc2_codes[i]
        name = SITC2_NAME.get(code, "")
        d = df[df["sitc2"] == code].sort_values("year")
        if d.empty:
            ax.set_title(f"{code}: {name}\n(no data)", fontsize=10)
            ax.axis("off")
            continue

        years = d["year"].to_numpy()

        if mode == "exports":
            I = d["I_exports"].to_numpy()
            lo = d["I_exports_ci95_norm_low"].to_numpy()
            hi = d["I_exports_ci95_norm_high"].to_numpy()
        else:
            I = d["I_imports"].to_numpy()
            lo = d["I_imports_ci95_norm_low"].to_numpy()
            hi = d["I_imports_ci95_norm_high"].to_numpy()

        ax.fill_between(years, lo, hi, alpha=0.25, linewidth=0)
        ax.plot(years, I, linewidth=1.8)
        ax.axhline(0, linestyle="--", linewidth=1, alpha=0.4)

        ax.set_title(f"{code}: {name}", fontsize=10)
        ax.set_ylim(*y_lim)
        ax.grid(True, alpha=0.2)

        # Keep x-axis readable
        xt = years[::5]  # every 5 years
        ax.set_xticks(xt)
        ax.tick_params(axis="x", labelrotation=90, labelsize=7)
        ax.tick_params(axis="y", labelsize=8)

    fig.suptitle(f"Moran's I – SITC-2 ({mode.capitalize()}) – 4×4 batch", y=0.995, fontsize=16)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

# -----------------------------
# 1) Individual graphs (one per SITC-2)
# -----------------------------
for code in SITC2_LIST:
    name = SITC2_NAME[code]
    d = df[df["sitc2"] == code].copy()
    if d.empty:
        continue
    fname = safe_filename(f"moran_sitc2_{code}_{name}.png")
    out_path = os.path.join(IND_DIR, fname)
    plot_two_panel_sector(d, code, name, out_path, Y_LIM)

print("Saved individual SITC-2 figures to:", IND_DIR)

# -----------------------------
# 2) 4×4 grouped figures (exports page(s) + imports page(s))
#    Each page contains 16 SITC-2 categories.
# -----------------------------
batch_size = 16
n_batches = math.ceil(len(SITC2_LIST) / batch_size)

for b in range(n_batches):
    batch_codes = SITC2_LIST[b*batch_size:(b+1)*batch_size]

    out_exp = os.path.join(GRID_DIR, f"GRID4x4_{b+1:02d}_exports.png")
    out_imp = os.path.join(GRID_DIR, f"GRID4x4_{b+1:02d}_imports.png")

    plot_grid_4x4(df, batch_codes, mode="exports", out_path=out_exp, y_lim=Y_LIM)
    plot_grid_4x4(df, batch_codes, mode="imports", out_path=out_imp, y_lim=Y_LIM)

print("Saved 4×4 grid figures to:", GRID_DIR)
print("Y_LIM used:", Y_LIM)


Saved individual SITC-2 figures to: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/figures_moran_sitc2/individual
Saved 4×4 grid figures to: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/figures_moran_sitc2/grids_4x4
Y_LIM used: (-0.024215979863066606, 0.2324948134946137)


In [ ]:
Trend analysis

In [ ]:
import os
import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
IN_CSV = os.path.join(BASE, "moran_S2_1976_2023_sitc2_normal_inference.csv")  # <-- adjust if needed
OUT_CSV = os.path.join(BASE, "moran_sitc2_pattern_classification.csv")

# Two periods
PERIODS = {
    "full": (1976, 2023),
    "2008_2023": (2008, 2023),
}

FLOWS = ["exports", "imports"]

# Rule parameters (as discussed)
R2_CONST_MAX = 0.10
DELTA_R2_CUTOFF = 0.05
TURN_MIN = 0.15
TURN_MAX = 0.85

# =========================
# Helpers: OLS fit for y ~ [1, tau] and y ~ [1, tau, tau^2]
# =========================
def _fit_ols(y: np.ndarray, X: np.ndarray):
    """
    Returns:
      beta (k,), yhat (n,), resid (n,), r2 (float)
    Uses least squares. Handles degenerate variance by returning r2=np.nan.
    """
    y = y.astype(float)
    # drop non-finite rows
    m = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
    y = y[m]
    X = X[m]
    n = len(y)
    if n < X.shape[1] + 1:
        return None, None, None, np.nan

    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    yhat = X @ beta
    resid = y - yhat

    sst = np.sum((y - np.mean(y)) ** 2)
    if sst <= 0:
        r2 = np.nan
    else:
        sse = np.sum(resid ** 2)
        r2 = 1 - (sse / sst)
    return beta, yhat, resid, float(r2)

def _tau_from_years(years: np.ndarray):
    years = years.astype(int)
    t0, tT = years.min(), years.max()
    if tT == t0:
        return np.zeros_like(years, dtype=float)
    return (years - t0) / (tT - t0)

def _turning_point(beta1: float, beta2: float):
    if not np.isfinite(beta1) or not np.isfinite(beta2) or beta2 == 0:
        return np.nan
    return -beta1 / (2.0 * beta2)

# =========================
# Load data
# =========================
df = pd.read_csv(IN_CSV)
df = df[df["status"] == "ok"].copy()
df["year"] = df["year"].astype(int)
df["sitc2"] = df["sitc2"].astype(str).str.zfill(2)

# Keep only needed columns defensively
need_cols = ["year", "sitc2", "I_exports", "I_imports"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

# =========================
# Compute model diagnostics per series
# =========================
records = []

for period_name, (ymin, ymax) in PERIODS.items():
    dper = df[(df["year"] >= ymin) & (df["year"] <= ymax)].copy()

    for flow in FLOWS:
        Icol = f"I_{flow}"

        # Fit each SITC2 separately
        for sitc2, dsec in dper.groupby("sitc2"):
            dsec = dsec.sort_values("year")
            years = dsec["year"].to_numpy()
            y = dsec[Icol].to_numpy(dtype=float)

            # Drop non-finite y
            mask = np.isfinite(y)
            years = years[mask]
            y = y[mask]

            n = len(y)
            if n < 6:
                # too few points to classify reliably
                records.append({
                    "sitc2": sitc2,
                    "flow": flow,
                    "period": period_name,
                    "n_years": int(n),
                    "pattern": "several fluctuations / no clear pattern",
                    "beta_lin": np.nan,
                    "r2_lin": np.nan,
                    "beta1_quad": np.nan,
                    "beta2_quad": np.nan,
                    "r2_quad": np.nan,
                    "delta_r2": np.nan,
                    "turning_point_tau": np.nan,
                    "notes": "too_few_years",
                })
                continue

            tau = _tau_from_years(years)
            X_lin = np.column_stack([np.ones_like(tau), tau])
            X_quad = np.column_stack([np.ones_like(tau), tau, tau**2])

            beta_lin, _, _, r2_lin = _fit_ols(y, X_lin)
            beta_quad, _, _, r2_quad = _fit_ols(y, X_quad)

            # Extract coefficients
            beta = float(beta_lin[1]) if beta_lin is not None else np.nan
            b1 = float(beta_quad[1]) if beta_quad is not None else np.nan
            b2 = float(beta_quad[2]) if beta_quad is not None else np.nan

            # Diagnostics
            delta_r2 = (r2_quad - r2_lin) if (np.isfinite(r2_quad) and np.isfinite(r2_lin)) else np.nan
            tau_star = _turning_point(b1, b2)

            records.append({
                "sitc2": sitc2,
                "flow": flow,
                "period": period_name,
                "n_years": int(n),
                "beta_lin": beta,
                "abs_beta_lin": abs(beta) if np.isfinite(beta) else np.nan,
                "r2_lin": r2_lin,
                "beta1_quad": b1,
                "beta2_quad": b2,
                "r2_quad": r2_quad,
                "delta_r2": float(delta_r2) if np.isfinite(delta_r2) else np.nan,
                "turning_point_tau": float(tau_star) if np.isfinite(tau_star) else np.nan,
            })

diag = pd.DataFrame(records)

# =========================
# Compute percentile thresholds (per flow x period)
# =========================
thresholds = []
for (period_name, flow), g in diag.groupby(["period", "flow"]):
    vals = g["abs_beta_lin"].to_numpy()
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        b10 = np.nan
        b25 = np.nan
    else:
        b10 = float(np.nanpercentile(vals, 10))
        b25 = float(np.nanpercentile(vals, 25))
    thresholds.append({"period": period_name, "flow": flow, "beta10": b10, "beta25": b25})

thr = pd.DataFrame(thresholds)

diag = diag.merge(thr, on=["period", "flow"], how="left")

# =========================
# Apply classification rules
# =========================
def classify_row(r):
    beta = r["beta_lin"]
    absb = r["abs_beta_lin"]
    r2_lin = r["r2_lin"]
    delta_r2 = r["delta_r2"]
    b2 = r["beta2_quad"]
    tau_star = r["turning_point_tau"]
    b10 = r["beta10"]
    b25 = r["beta25"]

    # Defensive: if core stats missing, call it fluctuating
    if not (np.isfinite(beta) and np.isfinite(absb) and np.isfinite(r2_lin) and np.isfinite(b10) and np.isfinite(b25)):
        return "several fluctuations / no clear pattern"

    # Step 1: Constant
    if absb <= b10 and r2_lin < R2_CONST_MAX:
        return "constant"

    # Step 2: Steady inc/dec (requires meaningful slope + little curvature gain)
    if absb >= b25 and np.isfinite(delta_r2) and delta_r2 < DELTA_R2_CUTOFF:
        return "steady increase" if beta > 0 else "steady decrease"

    # Step 3: U / inverted U (requires curvature gain + internal turning point)
    if np.isfinite(delta_r2) and delta_r2 >= DELTA_R2_CUTOFF and np.isfinite(b2) and b2 != 0 and np.isfinite(tau_star):
        if TURN_MIN <= tau_star <= TURN_MAX:
            return "u shape" if b2 > 0 else "inverted u shape"

    # Step 4: Fluctuating
    return "several fluctuations / no clear pattern"

diag["pattern"] = diag.apply(classify_row, axis=1)

# Add a simple notes field for transparency
def notes_row(r):
    notes = []
    if np.isfinite(r["r2_lin"]) and r["r2_lin"] >= 0.5:
        notes.append("strong_linear_fit")
    if np.isfinite(r["delta_r2"]) and r["delta_r2"] >= 0.10:
        notes.append("strong_quadratic_gain")
    if np.isfinite(r["turning_point_tau"]) and not (TURN_MIN <= r["turning_point_tau"] <= TURN_MAX):
        notes.append("turning_point_outside_core")
    return ";".join(notes)

diag["notes"] = diag.apply(notes_row, axis=1)

# Keep and order columns
out_cols = [
    "sitc2","flow","period","pattern","n_years",
    "beta_lin","r2_lin","beta1_quad","beta2_quad","r2_quad","delta_r2","turning_point_tau",
    "beta10","beta25","notes"
]
out = diag[out_cols].sort_values(["flow","sitc2","period"])

out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

# Quick sanity checks (counts)
print("\nCounts by period/flow/pattern:")
print(out.groupby(["period","flow","pattern"]).size().reset_index(name="n").sort_values(["period","flow","n"], ascending=[True,True,False]).to_string(index=False))

# Preview
out.head(20)


Saved: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/moran_sitc2_pattern_classification.csv

Counts by period/flow/pattern:
   period    flow                                 pattern  n
2008_2023 exports                         steady decrease 32
2008_2023 exports                        inverted u shape 11
2008_2023 exports                                 u shape 10
2008_2023 exports                                constant  7
2008_2023 exports several fluctuations / no clear pattern  3
2008_2023 imports                                 u shape 46
2008_2023 imports                         steady decrease  9
2008_2023 imports                                constant  7
2008_2023 imports                        inverted u shape  1
     full exports                        inverted u shape 19
     full exports                         steady increase 17
     full exports several fluctuations / no clear pattern 16
     full exports                                constant  7
     full expo

,sitc2,flow,period,pattern,n_years,beta_lin,r2_lin,beta1_quad,beta2_quad,r2_quad,delta_r2,turning_point_tau,beta10,beta25,notes
126,00,exports,2008_2023,u shape,16,-0.010250,0.140780,-0.039504,0.029254,0.226396,0.085616,0.675197,0.003532,0.006670,
0,00,exports,full,inverted u shape,48,0.025653,0.287840,0.070771,-0.045118,0.349644,0.061804,0.784286,0.004148,0.013757,
127,01,exports,2008_2023,steady decrease,16,-0.030282,0.544883,-0.052128,0.021846,0.566058,0.021174,1.193076,0.003532,0.006670,strong_linear_fit;turning_point_outside_core
1,01,exports,full,constant,48,0.003674,0.007398,0.027051,-0.023378,0.028192,0.020794,0.578574,0.004148,0.013757,
128,02,exports,2008_2023,steady decrease,16,-0.012481,0.227326,-0.028972,0.016492,0.256961,0.029635,0.878403,0.003532,0.006670,turning_point_outside_core
2,02,exports,full,inverted u shape,48,0.020512,0.197635,0.067100,-0.046589,0.268406,0.070771,0.720137,0.004148,0.013757,
129,03,exports,2008_2023,steady decrease,16,-0.007299,0.048941,-0.011493,0.004194,0.050148,0.001207,1.370133,0.003532,0.006670,turning_point_outside_core
3,03,exports,full,steady decrease,48,-0.045138,0.504491,-0.071516,0.026378,0.516450,0.011959,1.355584,0.004148,0.013757,strong_linear_fit;turning_point_outside_core
130,04,exports,2008_2023,several fluctuations / no clear pattern,16,-0.004756,0.041419,-0.018117,0.013361,0.065828,0.024409,0.677975,0.003532,0.006670,
4,04,exports,full,constant,48,0.002591,0.007095,-0.006899,0.009490,0.013704,0.006609,0.363503,0.004148,0.013757,


Heatmap trend

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
IN_CSV = os.path.join(BASE, "moran_S2_1976_2023_sitc2_normal_inference.csv")
FIG_DIR = os.path.join(BASE, "figures_directional_yearly")

os.makedirs(FIG_DIR, exist_ok=True)

START_YEAR = 1976
END_YEAR = 2023

# Colors
COLOR_MAP = {
    "growing": "#2ca02c",     # green
    "constant": "#ffeb3b",    # yellow
    "decreasing": "#d62728",  # red
}

# SITC-2 names (short)
SITC2_NAME = {
    "00":"Live animals","01":"Meat","02":"Dairy","03":"Fish","04":"Cereals",
    "05":"Fruit & veg","06":"Sugar","07":"Coffee/cocoa","08":"Feed","09":"Other food",
    "11":"Beverages","12":"Tobacco","21":"Hides","22":"Oil seeds","23":"Rubber",
    "24":"Wood","25":"Pulp","26":"Textile fibres","27":"Minerals","28":"Ores",
    "29":"Crude materials","32":"Coal","33":"Oil","34":"Gas","35":"Electricity",
    "41":"Animal oils","42":"Veg oils","43":"Prepared fats","51":"Organic chem",
    "52":"Inorganic chem","53":"Dyes","54":"Pharma","55":"Cosmetics",
    "56":"Fertilizers","57":"Plastics (prim)","58":"Plastics (art)",
    "59":"Chem n.e.s.","61":"Leather","62":"Rubber manuf","63":"Wood manuf",
    "64":"Paper","65":"Textiles","66":"Non-metallic","67":"Iron & steel",
    "68":"Non-ferrous","69":"Metal manuf","71":"Power mach","72":"Spec mach",
    "73":"Metal mach","74":"Gen mach","75":"Office mach","76":"Telecom",
    "77":"Electrical","78":"Vehicles","79":"Transport","81":"Prefabs",
    "82":"Furniture","83":"Travel goods","84":"Apparel","85":"Footwear",
    "87":"Optical","88":"Misc manuf","89":"Unclassified"
}

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(IN_CSV)
df = df[df["status"] == "ok"].copy()
df["year"] = df["year"].astype(int)
df["sitc2"] = df["sitc2"].astype(str).str.zfill(2)

df = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)]

# =========================
# YEARLY DIFFERENCES
# =========================
records = []

for flow in ["exports", "imports"]:
    col = f"I_{flow}"

    for sitc2, g in df.groupby("sitc2"):
        g = g.sort_values("year")
        years = g["year"].to_numpy()
        I = g[col].to_numpy(dtype=float)

        dI = np.diff(I)
        dy = years[1:]

        for y, di in zip(dy, dI):
            records.append({
                "sitc2": sitc2,
                "flow": flow,
                "year": y,
                "delta_I": di,
                "abs_delta_I": abs(di),
            })

chg = pd.DataFrame(records)

# =========================
# THRESHOLDS (per flow)
# =========================
out = []

for flow, g in chg.groupby("flow"):
    eps = np.nanpercentile(g["abs_delta_I"], 10)

    for _, r in g.iterrows():
        if abs(r["delta_I"]) <= eps:
            direction = "constant"
        elif r["delta_I"] > 0:
            direction = "growing"
        else:
            direction = "decreasing"

        out.append({
            "sitc2": r["sitc2"],
            "flow": flow,
            "year": r["year"],
            "direction": direction,
        })

out = pd.DataFrame(out)

# =========================
# HEATMAPS (one cell per year)
# =========================
for flow in ["exports", "imports"]:
    h = out[out["flow"] == flow]
    h = h.pivot(index="sitc2", columns="year", values="direction")
    h = h.loc[sorted(h.index)]

    fig, ax = plt.subplots(figsize=(18, 14))

    for i, sitc2 in enumerate(h.index):
        for j, year in enumerate(h.columns):
            d = h.loc[sitc2, year]
            if pd.isna(d):
                continue
            ax.add_patch(
                plt.Rectangle(
                    (j, i), 1, 1,
                    color=COLOR_MAP[d]
                )
            )

    ax.set_xlim(0, len(h.columns))
    ax.set_ylim(len(h.index), 0)

    ax.set_xticks(np.arange(len(h.columns)) + 0.5)
    ax.set_xticklabels(h.columns, rotation=90)

    ax.set_yticks(np.arange(len(h.index)) + 0.5)
    ax.set_yticklabels(
        [f"{s} – {SITC2_NAME.get(s,'')}" for s in h.index]
    )

    ax.set_title(f"Yearly direction of Moran’s I ({flow.capitalize()})")
    plt.tight_layout()
    plt.savefig(
        os.path.join(FIG_DIR, f"heatmap_direction_yearly_{flow}.png"),
        dpi=200
    )
    plt.close()

print("Yearly directional heatmaps saved in:", FIG_DIR)


Yearly directional heatmaps saved in: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/figures_directional_yearly


Shede Heat Map Moran I

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
IN_CSV = os.path.join(BASE, "moran_S2_1976_2023_sitc2_normal_inference.csv")
FIG_DIR = os.path.join(BASE, "figures_level_heatmaps")
os.makedirs(FIG_DIR, exist_ok=True)

START_YEAR = 1976
END_YEAR = 2023

# SITC-2 names (short)
SITC2_NAME = {
    "00":"Live animals","01":"Meat","02":"Dairy","03":"Fish","04":"Cereals",
    "05":"Fruit & veg","06":"Sugar","07":"Coffee/cocoa","08":"Feed","09":"Other food",
    "11":"Beverages","12":"Tobacco","21":"Hides","22":"Oil seeds","23":"Rubber",
    "24":"Wood","25":"Pulp","26":"Textile fibres","27":"Minerals","28":"Ores",
    "29":"Crude materials","32":"Coal","33":"Oil","34":"Gas","35":"Electricity",
    "41":"Animal oils","42":"Veg oils","43":"Prepared fats","51":"Organic chem",
    "52":"Inorganic chem","53":"Dyes","54":"Pharma","55":"Cosmetics",
    "56":"Fertilizers","57":"Plastics (prim)","58":"Plastics (art)",
    "59":"Chem n.e.s.","61":"Leather","62":"Rubber manuf","63":"Wood manuf",
    "64":"Paper","65":"Textiles","66":"Non-metallic","67":"Iron & steel",
    "68":"Non-ferrous","69":"Metal manuf","71":"Power mach","72":"Spec mach",
    "73":"Metal mach","74":"Gen mach","75":"Office mach","76":"Telecom",
    "77":"Electrical","78":"Vehicles","79":"Transport","81":"Prefabs",
    "82":"Furniture","83":"Travel goods","84":"Apparel","85":"Footwear",
    "87":"Optical","88":"Misc manuf","89":"Unclassified"
}

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(IN_CSV)
df = df[df["status"] == "ok"].copy()
df["year"] = df["year"].astype(int)
df["sitc2"] = df["sitc2"].astype(str).str.zfill(2)
df = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)]

# =========================
# PLOTTERS
# =========================
def plot_level_heatmap(mat: pd.DataFrame, title: str, outpath: str, cmap: str, vmin=None, vmax=None):
    """
    mat: index=sitc2, columns=year, values=Moran I
    """
    fig, ax = plt.subplots(figsize=(18, 14))
    arr = mat.to_numpy(dtype=float)

    im = ax.imshow(arr, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("SITC-2 sector")

    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=90)

    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_yticklabels([f"{s} – {SITC2_NAME.get(s,'')}" for s in mat.index])

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Moran's I")

    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

# =========================
# BUILD MATRICES + SCALES
# =========================
for flow in ["exports", "imports"]:
    col = f"I_{flow}"

    mat = df.pivot(index="sitc2", columns="year", values=col)
    mat = mat.loc[sorted(mat.index)]

    # Global scaling for this flow
    vmin = float(np.nanmin(mat.to_numpy()))
    vmax = float(np.nanmax(mat.to_numpy()))

    # Variant A: sequential (Blues)
    plot_level_heatmap(
        mat,
        title=f"Moran's I level heatmap ({flow.capitalize()}) – Sequential scale",
        outpath=os.path.join(FIG_DIR, f"heatmap_level_{flow}_sequential.png"),
        cmap="Blues",
        vmin=vmin, vmax=vmax
    )

    # Variant B: diverging around 0 (more faithful if negatives matter)
    bound = float(max(abs(vmin), abs(vmax)))
    plot_level_heatmap(
        mat,
        title=f"Moran's I level heatmap ({flow.capitalize()}) – Diverging around 0",
        outpath=os.path.join(FIG_DIR, f"heatmap_level_{flow}_diverging.png"),
        cmap="RdBu_r",
        vmin=-bound, vmax=bound
    )

print("Saved level heatmaps to:", FIG_DIR)


Saved level heatmaps to: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/figures_level_heatmaps


3 Digit Analysis

In [ ]:
# ============================================================
# Moran's I by YEAR × SITC-3 (NORMAL-ASSUMPTION inference)
#   - Global exports and imports
#   - Keeps ALL countries in OD universe (missing trade -> 0)
#   - Uses log10(value + 1)
#   - Outputs (per year, sitc3, flow):
#       n, I, EI, VI_norm, SE_norm, z_norm, p_norm_2s, CI95_low/high
#
# Data:
#   OD_Matrix.csv
#   S2_YYYY.parquet  (1976–2023)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install duckdb libpysal esda scipy

import os
import numpy as np
import pandas as pd
import duckdb

from esda.moran import Moran
from libpysal.weights import W as psW
from scipy.stats import norm

# ----------------------------
# Config (EDIT THESE)
# ----------------------------
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
OD_PATH = os.path.join(BASE, "OD_Matrix.csv")

PREFIX = "S2"
START_YEAR = 1976
END_YEAR   = 2023

ALPHA = 0.05           # 95% CI
PERMUTATIONS = 0       # keep 0 unless you want p_sim/z_sim too (optional)

EXPORTER_COL = "exporter"
IMPORTER_COL = "importer"
COMMODITY_COL = "commoditycode"
VALUE_EXPORT_COL = "value_exporter"
VALUE_IMPORT_COL = "value_importer"

OUT_CSV = os.path.join(BASE, f"moran_{PREFIX}_{START_YEAR}_{END_YEAR}_sitc3_normal_inference.csv")

# ----------------------------
# SITC-3 whitelist (YOUR LIST)
# ----------------------------
SITC3_CODES = {
    "001","011","012","016","017","022","023","024","025",
    "034","035","036","037",
    "041","042","043","044","045","046","047","048",
    "054","056",
    "061","062",
    "071","072","073","074",
    "081","098",
    "111",
    "121","122",
    "211","212",
    "221","222","223",
    "231",
    "241","242",
    "261","262","263","264","265","266","267","268","269",
    "271","272","273","274","275","276","277","278",
    "281","282","283","284","285","286","287","288","289",
    "291",
    "321","322","325",
    "333","334","335",
    "341","342","343","344","345","347",
    "351",
    "411","412","421","431","442",
    "511","512","513","514","515","516",
    "522","523","524","525",
    "531","532","533",
    "541","542",
    "551","553","554",
    "561","562",
    "571","572","573","574","575",
    "581","582","583",
    "591","592","593","597","598",
    "611","612","613",
    "621","625","629",
    "633","634","635",
    "641","642",
    "651",
    "661","662","663","664","665","667","668","679","699",
    "711","712","713","714",
    "721","722","723","724","725","728",
    "731","732","733","734","735",
    "741","742","743","744","745","746","747","748",
    "751","752","759",
    "761","762","763","764",
    "771","772","773","774","775",
    "781","782","783","784","785","786",
    "791","792","793",
    "811","821","831",
    "841","842","843","844","845","846",
    "851",
    "871","881",
    "891","892","893","894","898",
    "961","971"
}

SITC3_LIST_SQL = ",".join([f"'{c}'" for c in sorted(SITC3_CODES)])

# ----------------------------
# Load OD once -> build W_full once (inverse-distance, row-standardized)
# ----------------------------
od_long = pd.read_csv(OD_PATH)
od_long["origin"] = od_long["origin"].astype(str).str.upper()
od_long["destination"] = od_long["destination"].astype(str).str.upper()

D_full = od_long.pivot(index="origin", columns="destination", values="distance_km")
all_codes = sorted(set(D_full.index) | set(D_full.columns))
D_full = D_full.reindex(index=all_codes, columns=all_codes)

# diagonal
np.fill_diagonal(D_full.values, 0.0)

D = D_full.to_numpy(dtype=float)
np.fill_diagonal(D, np.nan)

W_dense = 1.0 / D
W_dense = W_dense / np.nansum(W_dense, axis=1, keepdims=True)

# build PySAL W once for the full OD universe
neighbors, weights = {}, {}
for i, c in enumerate(all_codes):
    js = np.where(np.isfinite(W_dense[i]) & (W_dense[i] > 0))[0]
    neighbors[c] = [all_codes[j] for j in js]
    weights[c] = [float(W_dense[i, j]) for j in js]

W_full = psW(neighbors, weights)
W_full.transform = "R"

# ----------------------------
# Helpers
# ----------------------------
def log10_1p(x):
    return np.log10(np.asarray(x, dtype=float) + 1.0)

def agg_year_trade_sitc3(parquet_path: str):
    """
    Returns two dataframes:
      exports: ISO_A3, sitc3, exports
      imports: ISO_A3, sitc3, imports
    Restricted to SITC3_CODES.
    """
    con = duckdb.connect()

    exp = con.execute(f"""
        SELECT
            UPPER({EXPORTER_COL}) AS ISO_A3,
            SUBSTR(CAST({COMMODITY_COL} AS VARCHAR), 1, 3) AS sitc3,
            SUM({VALUE_EXPORT_COL}) AS exports
        FROM read_parquet('{parquet_path}')
        GROUP BY 1,2
    """).df()

    imp = con.execute(f"""
        SELECT
            UPPER({IMPORTER_COL}) AS ISO_A3,
            SUBSTR(CAST({COMMODITY_COL} AS VARCHAR), 1, 3) AS sitc3,
            SUM({VALUE_IMPORT_COL}) AS imports
        FROM read_parquet('{parquet_path}')
        GROUP BY 1,2
    """).df()

    con.close()

    exp["sitc3"] = exp["sitc3"].astype(str).str.zfill(3)
    imp["sitc3"] = imp["sitc3"].astype(str).str.zfill(3)

    exp = exp[exp["sitc3"].isin(SITC3_CODES)].copy()
    imp = imp[imp["sitc3"].isin(SITC3_CODES)].copy()

    return exp, imp

def moran_normal_stats_full_universe(values_by_code: dict, alpha=0.05, permutations=0):
    """
    Moran's I with normal-assumption inference using FULL OD universe:
      - codes fixed to all_codes
      - missing codes -> 0
      - values transformed outside (we will feed log10(1+value))

    values_by_code: dict {ISO_A3: value} aligned to all_codes universe
    """
    # align to full universe and ensure numeric
    vals = np.array([float(values_by_code.get(c, 0.0)) for c in all_codes], dtype=float)

    n = len(vals)
    if n < 5 or np.std(vals) == 0:
        out = {
            "n": int(n),
            "I": np.nan,
            "EI": np.nan,
            "VI_norm": np.nan,
            "SE_norm": np.nan,
            "z_norm": np.nan,
            "p_norm_2s": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
        }
        if permutations and permutations > 0:
            out.update({"p_sim": np.nan, "z_sim": np.nan})
        return out

    m = Moran(vals, W_full, permutations=permutations)

    I = float(m.I)
    EI = float(m.EI)
    VI = float(m.VI_norm) if np.isfinite(m.VI_norm) else np.nan
    SE = float(np.sqrt(VI)) if np.isfinite(VI) and VI >= 0 else np.nan

    z_norm = float(m.z_norm) if hasattr(m, "z_norm") and np.isfinite(m.z_norm) else (
        (I - EI) / SE if np.isfinite(EI) and np.isfinite(SE) and SE > 0 else np.nan
    )
    p_norm = float(2 * (1 - norm.cdf(abs(z_norm)))) if np.isfinite(z_norm) else np.nan

    zcrit = norm.ppf(1 - alpha/2)
    ci_low = float(I - zcrit * SE) if np.isfinite(SE) else np.nan
    ci_high = float(I + zcrit * SE) if np.isfinite(SE) else np.nan

    out = {
        "n": int(n),
        "I": I,
        "EI": EI,
        "VI_norm": VI,
        "SE_norm": SE,
        "z_norm": z_norm,
        "p_norm_2s": p_norm,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
    }
    if permutations and permutations > 0:
        out.update({"p_sim": float(m.p_sim), "z_sim": float(m.z_sim)})

    return out

# ----------------------------
# Run
# ----------------------------
rows = []

sitc3_sorted = sorted(SITC3_CODES)

for y in range(START_YEAR, END_YEAR + 1):
    fp = os.path.join(BASE, f"{PREFIX}_{y}.parquet")
    if not os.path.exists(fp):
        # still write missing year status rows (optional)
        for sitc3 in sitc3_sorted:
            rows.append({"year": y, "sitc3": sitc3, "status": "missing_file"})
        continue

    exp_df, imp_df = agg_year_trade_sitc3(fp)

    # pre-split per sitc3 for speed
    exp_groups = {k: g for k, g in exp_df.groupby("sitc3")}
    imp_groups = {k: g for k, g in imp_df.groupby("sitc3")}

    for sitc3 in sitc3_sorted:
        # ---- exports dict (missing countries -> 0)
        if sitc3 in exp_groups:
            g = exp_groups[sitc3]
            d_exp = dict(zip(g["ISO_A3"].tolist(), g["exports"].astype(float).tolist()))
        else:
            d_exp = {}

        # ---- imports dict
        if sitc3 in imp_groups:
            g = imp_groups[sitc3]
            d_imp = dict(zip(g["ISO_A3"].tolist(), g["imports"].astype(float).tolist()))
        else:
            d_imp = {}

        # transform values into log10(1+value) but keep zeros included
        d_exp_t = {c: float(log10_1p(v)) for c, v in d_exp.items()}
        d_imp_t = {c: float(log10_1p(v)) for c, v in d_imp.items()}
        # missing codes automatically treated as 0.0 => log10(1)=0 in moran fn

        exp_stats = moran_normal_stats_full_universe(d_exp_t, alpha=ALPHA, permutations=PERMUTATIONS)
        imp_stats = moran_normal_stats_full_universe(d_imp_t, alpha=ALPHA, permutations=PERMUTATIONS)

        row = {
            "year": y,
            "sitc3": sitc3,
            "status": "ok",

            # exports
            "n_exports": exp_stats["n"],
            "I_exports": exp_stats["I"],
            "EI_exports": exp_stats["EI"],
            "VI_norm_exports": exp_stats["VI_norm"],
            "SE_norm_exports": exp_stats["SE_norm"],
            "z_norm_exports": exp_stats["z_norm"],
            "p_norm_2s_exports": exp_stats["p_norm_2s"],
            "I_exports_ci95_norm_low": exp_stats["ci95_low"],
            "I_exports_ci95_norm_high": exp_stats["ci95_high"],

            # imports
            "n_imports": imp_stats["n"],
            "I_imports": imp_stats["I"],
            "EI_imports": imp_stats["EI"],
            "VI_norm_imports": imp_stats["VI_norm"],
            "SE_norm_imports": imp_stats["SE_norm"],
            "z_norm_imports": imp_stats["z_norm"],
            "p_norm_2s_imports": imp_stats["p_norm_2s"],
            "I_imports_ci95_norm_low": imp_stats["ci95_low"],
            "I_imports_ci95_norm_high": imp_stats["ci95_high"],
        }

        if PERMUTATIONS and PERMUTATIONS > 0:
            row.update({
                "p_sim_exports": exp_stats["p_sim"],
                "z_sim_exports": exp_stats["z_sim"],
                "p_sim_imports": imp_stats["p_sim"],
                "z_sim_imports": imp_stats["z_sim"],
            })

        rows.append(row)

out = pd.DataFrame(rows).sort_values(["year", "sitc3"])
out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
out.head(20)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/moran_S2_1976_2023_sitc3_normal_inference.csv


,year,sitc3,status,n_exports,I_exports,EI_exports,VI_norm_exports,SE_norm_exports,z_norm_exports,p_norm_2s_exports,...,I_exports_ci95_norm_high,n_imports,I_imports,EI_imports,VI_norm_imports,SE_norm_imports,z_norm_imports,p_norm_2s_imports,I_imports_ci95_norm_low,I_imports_ci95_norm_high
0,1976,001,ok,171,0.038525,-0.005882,0.000122,0.011057,4.016126,5.916255e-05,...,0.060197,171,0.043942,-0.005882,0.000122,0.011057,4.505973,6.606939e-06,0.022270,0.065614
1,1976,011,ok,171,0.093651,-0.005882,0.000122,0.011057,9.001575,0.000000e+00,...,0.115323,171,0.039319,-0.005882,0.000122,0.011057,4.087928,4.352432e-05,0.017647,0.060991
2,1976,012,ok,171,0.066253,-0.005882,0.000122,0.011057,6.523742,6.857470e-11,...,0.087925,171,0.082086,-0.005882,0.000122,0.011057,7.955673,1.776357e-15,0.060414,0.103758
3,1976,016,ok,171,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1976,017,ok,171,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1976,022,ok,171,0.064742,-0.005882,0.000122,0.011057,6.387093,1.690688e-10,...,0.086414,171,0.048365,-0.005882,0.000122,0.011057,4.905968,9.296765e-07,0.026693,0.070036
6,1976,023,ok,171,0.054778,-0.005882,0.000122,0.011057,5.486007,4.111200e-08,...,0.076450,171,0.026044,-0.005882,0.000122,0.011057,2.887319,3.885405e-03,0.004372,0.047716
7,1976,024,ok,171,0.085594,-0.005882,0.000122,0.011057,8.272880,2.220446e-16,...,0.107266,171,0.040078,-0.005882,0.000122,0.011057,4.156524,3.231266e-05,0.018406,0.061750
8,1976,025,ok,171,0.043732,-0.005882,0.000122,0.011057,4.486989,7.223677e-06,...,0.065404,171,0.064169,-0.005882,0.000122,0.011057,6.335255,2.369491e-10,0.042497,0.085841
9,1976,034,ok,171,0.065138,-0.005882,0.000122,0.011057,6.422882,1.337186e-10,...,0.086809,171,0.044645,-0.005882,0.000122,0.011057,4.569593,4.886733e-06,0.022973,0.066317


In [ ]:
# ============================================================
# Moran's I LEVEL HEATMAPS (SITC-3) — Exports & Imports
#   - Reads: moran_S2_1976_2023_sitc3_normal_inference.csv
#   - Produces 4 figures:
#       exports: sequential + diverging
#       imports: sequential + diverging
#   - NaNs (undefined Moran's I) shown in light gray
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Config (EDIT THESE)
# ----------------------------
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"

IN_CSV = os.path.join(BASE, "moran_S2_1976_2023_sitc3_normal_inference.csv")
FIG_DIR = os.path.join(BASE, "figures_level_heatmaps_sitc3")

START_YEAR = 1976
END_YEAR   = 2023

os.makedirs(FIG_DIR, exist_ok=True)

# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(IN_CSV)
df = df[df["status"] == "ok"].copy()

df["year"] = df["year"].astype(int)
df["sitc3"] = df["sitc3"].astype(str).str.zfill(3)
df = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)]

# Stable ordering
sitc3_order = sorted(df["sitc3"].unique())
year_order = list(range(START_YEAR, END_YEAR + 1))

# ----------------------------
# Plot helper (NaNs in light gray)
# ----------------------------
def plot_level_heatmap(mat: pd.DataFrame, title: str, outpath: str, cmap: str, vmin=None, vmax=None):
    """
    mat: index=sitc3, columns=year, values=Moran I
    NaNs are rendered in light gray.
    """
    mat = mat.reindex(index=sitc3_order, columns=year_order)

    # dynamic height (SITC-3 can be many rows)
    fig_h = max(10, 0.16 * len(mat.index))
    fig, ax = plt.subplots(figsize=(18, fig_h))

    arr = mat.to_numpy(dtype=float)

    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad(color="#e6e6e6")  # NaN color

    im = ax.imshow(arr, aspect="auto", cmap=cmap_obj, vmin=vmin, vmax=vmax)

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("SITC-3 code")

    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=90)

    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_yticklabels(mat.index)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Moran's I")

    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

# ----------------------------
# Build matrices + plot
# ----------------------------
for flow in ["exports", "imports"]:
    col = f"I_{flow}"

    mat = df.pivot(index="sitc3", columns="year", values=col)
    mat = mat.reindex(index=sitc3_order, columns=year_order)

    # Global scaling for this flow (ignore NaNs)
    vmin = float(np.nanmin(mat.to_numpy()))
    vmax = float(np.nanmax(mat.to_numpy()))

    # Variant A: sequential scale (min..max)
    plot_level_heatmap(
        mat,
        title=f"Moran's I level heatmap (SITC-3, {flow.capitalize()}) — Sequential scale",
        outpath=os.path.join(FIG_DIR, f"heatmap_level_sitc3_{flow}_sequential.png"),
        cmap="Blues",
        vmin=vmin,
        vmax=vmax
    )

    # Variant B: diverging around 0 (symmetric)
    bound = float(max(abs(vmin), abs(vmax)))
    plot_level_heatmap(
        mat,
        title=f"Moran's I level heatmap (SITC-3, {flow.capitalize()}) — Diverging around 0",
        outpath=os.path.join(FIG_DIR, f"heatmap_level_sitc3_{flow}_diverging.png"),
        cmap="RdBu_r",
        vmin=-bound,
        vmax=bound
    )

print("Saved SITC-3 level heatmaps to:", FIG_DIR)


Saved SITC-3 level heatmaps to: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/figures_level_heatmaps_sitc3


In [ ]:
# ============================================================
# Moran's I LEVEL HEATMAPS (SITC-3) — CORRECTED LABELS
#   - Exports & Imports
#   - NaNs shown in light gray
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Config
# ----------------------------
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"

IN_CSV = os.path.join(BASE, "moran_S2_1976_2023_sitc3_normal_inference.csv")
FIG_DIR = os.path.join(BASE, "figures_level_heatmaps_sitc3_labeled")

START_YEAR = 1976
END_YEAR   = 2023

os.makedirs(FIG_DIR, exist_ok=True)

# ----------------------------
# CORRECTED SITC-3 LABEL MAP
# ----------------------------
SITC3_LABELS = {
    "001":"Live animals",
    "011":"Fresh beef",
    "012":"Other fresh meat",
    "016":"Cured meat",
    "017":"Processed meat",
    "022":"Dairy (non-butter/cheese)",
    "023":"Butter & milk fats",
    "024":"Cheese",
    "025":"Eggs",
    "034":"Fresh fish",
    "035":"Processed fish",
    "036":"Crustaceans & molluscs",
    "037":"Prepared seafood",
    "041":"Wheat",
    "042":"Rice",
    "043":"Barley",
    "044":"Maize",
    "045":"Other cereals",
    "046":"Wheat flour",
    "047":"Other cereal flours",
    "048":"Cereal preparations",
    "054":"Fresh vegetables",
    "056":"Prepared vegetables",
    "057":"Fresh fruit & nuts",
    "058":"Preserved fruit",
    "059":"Fruit & veg juices",
    "061":"Sugar & honey",
    "062":"Confectionery",
    "071":"Coffee",
    "072":"Cocoa",
    "073":"Chocolate products",
    "074":"Tea & mate",
    "075":"Spices",
    "081":"Animal feed",
    "091":"Margarine",
    "098":"Other food products",
    "111":"Soft drinks",
    "112":"Alcoholic beverages",
    "121":"Raw tobacco",
    "122":"Manufactured tobacco",
    "211":"Raw hides & skins",
    "212":"Raw furskins",
    "222":"Oilseeds (soft oils)",
    "223":"Oilseeds (other oils)",
    "231":"Natural rubber",
    "232":"Synthetic rubber",
    "244":"Raw cork",
    "245":"Fuel wood & charcoal",
    "246":"Wood waste & chips",
    "247":"Rough wood",
    "248":"Worked wood",
    "251":"Pulp & waste paper",
    "261":"Silk fibers",
    "263":"Cotton fibers",
    "264":"Jute & bast fibers",
    "265":"Other vegetable fibers",
    "266":"Synthetic fibers",
    "267":"Man-made fibers",
    "268":"Wool & animal hair",
    "269":"Used clothing & rags",
    "272":"Crude fertilizer",
    "273":"Stone & gravel",
    "274":"Sulfur & pyrites",
    "277":"Abrasives",
    "278":"Other crude minerals",
    "281":"Iron ore",
    "282":"Ferrous scrap",
    "283":"Copper ores",
    "284":"Nickel ores",
    "285":"Aluminum ores",
    "286":"Uranium & thorium ores",
    "287":"Other base metal ores",
    "288":"Non-ferrous scrap",
    "289":"Precious metal ores",
    "291":"Crude animal materials",
    "292":"Crude vegetable materials",
    "321":"Coal",
    "322":"Lignite & peat",
    "325":"Coke & semicoke",
    "333":"Crude oil",
    "334":"Refined petroleum",
    "335":"Residual petroleum",
    "342":"LPG",
    "343":"Natural gas",
    "344":"Other petroleum gases",
    "345":"Manufactured gas",
    "351":"Electricity",
    "411":"Animal fats",
    "421":"Vegetable oils (soft)",
    "422":"Vegetable oils (other)",
    "431":"Processed fats & waxes",
    "511":"Hydrocarbons",
    "512":"Alcohols & phenols",
    "513":"Carboxylic acids",
    "514":"Nitrogen compounds",
    "515":"Complex organics",
    "516":"Other organic chemicals",
    "522":"Inorganic chemicals",
    "523":"Inorganic salts",
    "524":"Other inorganic chemicals",
    "525":"Radioactive materials",
    "531":"Synthetic dyes",
    "532":"Tanning extracts",
    "533":"Paints & pigments",
    "541":"Pharma inputs",
    "542":"Medicines",
    "551":"Essential oils",
    "553":"Cosmetics",
    "554":"Soaps & cleaners",
    "562":"Fertilizers",
    "571":"Polyethylene",
    "572":"Polystyrene",
    "573":"PVC & halogen polymers",
    "574":"Engineering plastics",
    "575":"Other plastics",
    "579":"Plastic scrap",
    "581":"Plastic pipes",
    "582":"Plastic sheets",
    "583":"Plastic profiles",
    "591":"Agro-chemicals",
    "592":"Starches & glues",
    "593":"Explosives",
    "597":"Lubricants & additives",
    "598":"Other chemical products",
    "599":"Chemical residues & waste",
    "611":"Leather",
    "612":"Leather manufactures",
    "613":"Dressed furs",
    "621":"Rubber materials",
    "625":"Rubber tires",
    "629":"Rubber articles",
    "633":"Cork products",
    "634":"Engineered wood",
    "635":"Wood manufactures",
    "641":"Paper & board",
    "642":"Paper articles",
    "651":"Textile yarn",
    "652":"Cotton fabrics",
    "653":"Man-made fabrics",
    "654":"Other woven fabrics",
    "655":"Knitted fabrics",
    "656":"Trimmings & lace",
    "657":"Special textiles",
    "658":"Made-up textiles",
    "659":"Floor coverings",
    "661":"Cement & lime products",
    "662":"Clay & refractories",
    "663":"Other mineral manufactures",
    "664":"Glass",
    "665":"Glassware",
    "666":"Pottery",
    "667":"Precious stones",
    "671":"Pig iron & ferroalloys",
    "672":"Steel semifinished",
    "673":"Steel flat products (plain)",
    "674":"Steel flat products (coated)",
    "675":"Alloy steel flats",
    "676":"Steel bars & sections",
    "677":"Rails & track",
    "678":"Steel wire",
    "679":"Steel pipes",
    "681":"Precious metals",
    "682":"Copper metal",
    "683":"Nickel metal",
    "684":"Aluminum metal",
    "685":"Lead",
    "686":"Zinc",
    "687":"Tin",
    "689":"Other base metals",
    "691":"Metal structures",
    "692":"Metal containers",
    "693":"Wire products",
    "694":"Fasteners",
    "695":"Tools",
    "696":"Cutlery",
    "697":"Metal household goods",
    "699":"Other metal products",
    "711":"Boilers",
    "712":"Steam turbines",
    "713":"Internal combustion engines",
    "714":"Other engines",
    "716":"Electric generators",
    "718":"Power machinery",
    "721":"Agricultural machinery",
    "722":"Tractors",
    "723":"Construction machinery",
    "724":"Textile machinery",
    "725":"Paper machinery",
    "726":"Printing machinery",
    "727":"Food-processing machinery",
    "728":"Specialized machinery",
    "731":"Machine tools (cutting)",
    "733":"Machine tools (forming)",
    "735":"Machine tool parts",
    "737":"Metalworking machinery",
    "741":"Heating & cooling equipment",
    "742":"Liquid pumps",
    "743":"Compressors & fans",
    "744":"Handling equipment",
    "745":"Other machinery",
    "746":"Bearings",
    "747":"Valves",
    "748":"Transmission equipment",
    "749":"Machinery parts",
    "751":"Office machines",
    "752":"Computers & ADP",
    "759":"Office machine parts",
    "761":"Television equipment",
    "762":"Radio receivers",
    "763":"Audio-video equipment",
    "764":"Telecom equipment",
    "771":"Electric power equipment",
    "772":"Electrical switching gear",
    "773":"Electric distribution equipment",
    "774":"Medical diagnostic equipment",
    "775":"Household appliances",
    "776":"Semiconductors & ICs",
    "778":"Other electrical machinery",
    "781":"Passenger vehicles",
    "782":"Goods vehicles",
    "783":"Other vehicles",
    "784":"Vehicle parts",
    "785":"Motorcycles & bicycles",
    "786":"Trailers & containers",
    "791":"Railway vehicles",
    "792":"Aircraft & spacecraft",
    "793":"Ships & boats",
    "811":"Prefabricated buildings",
    "812":"Sanitary fixtures",
    "813":"Lighting fixtures",
    "821":"Furniture",
    "831":"Travel goods",
    "841":"Men’s woven apparel",
    "842":"Women’s woven apparel",
    "843":"Men’s knit apparel",
    "844":"Women’s knit apparel",
    "845":"Other apparel",
    "846":"Textile accessories",
    "848":"Non-textile apparel",
    "851":"Footwear",
    "871":"Optical instruments",
    "872":"Medical instruments",
    "873":"Meters & counters",
    "874":"Measuring instruments",
    "881":"Photographic equipment",
    "882":"Photo & cinema supplies",
    "883":"Developed film",
    "884":"Other optical goods",
    "885":"Watches & clocks",
    "891":"Arms & ammunition",
    "892":"Printed matter",
    "893":"Plastic articles",
    "894":"Toys & sports goods",
    "895":"Office supplies",
    "896":"Art & antiques",
    "897":"Jewelry",
    "898":"Musical instruments",
    "899":"Other manufactures",
    "931":"Special transactions",
    "950":"Gold coin",
    "961":"Non-gold coin",
    "971":"Non-monetary gold",
    "984":"Low-value imports",
    "992":"Low-value exports",
    "994":"Low-value Canada trade"
}

# ----------------------------
# Load and label data
# ----------------------------
df = pd.read_csv(IN_CSV)
df = df[df["status"] == "ok"].copy()

df["year"] = df["year"].astype(int)
df["sitc3"] = df["sitc3"].astype(str).str.zfill(3)

df = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)]

df["sitc3_label"] = df["sitc3"].apply(
    lambda x: f"{x} – {SITC3_LABELS.get(x, 'Unknown')}"
)

row_order = sorted(df["sitc3_label"].unique())
year_order = list(range(START_YEAR, END_YEAR + 1))

# ----------------------------
# Plot helper
# ----------------------------
def plot_heatmap(mat, title, outpath, cmap, vmin, vmax):
    fig, ax = plt.subplots(figsize=(20, max(12, 0.18 * len(mat))))
    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad("#e6e6e6")

    im = ax.imshow(mat.values, aspect="auto", cmap=cmap_obj, vmin=vmin, vmax=vmax)

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("SITC-3 product")

    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=90)

    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Moran’s I")

    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

# ----------------------------
# Build and plot
# ----------------------------
for flow in ["exports", "imports"]:
    col = f"I_{flow}"
    mat = df.pivot(index="sitc3_label", columns="year", values=col)
    mat = mat.reindex(index=row_order, columns=year_order)

    vmin = np.nanmin(mat.values)
    vmax = np.nanmax(mat.values)
    bound = max(abs(vmin), abs(vmax))

    plot_heatmap(
        mat,
        f"Moran’s I (SITC-3, {flow.capitalize()}) — Sequential scale",
        os.path.join(FIG_DIR, f"heatmap_level_sitc3_{flow}_sequential_labeled.png"),
        "Blues",
        vmin,
        vmax
    )

    plot_heatmap(
        mat,
        f"Moran’s I (SITC-3, {flow.capitalize()}) — Diverging around zero",
        os.path.join(FIG_DIR, f"heatmap_level_sitc3_{flow}_diverging_labeled.png"),
        "RdBu_r",
        -bound,
        bound
    )

print("Saved corrected labeled SITC-3 heatmaps to:", FIG_DIR)


Saved corrected labeled SITC-3 heatmaps to: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/figures_level_heatmaps_sitc3_labeled
